# WorldQuant Alpha101 - RESSET适配版

## tl;dr

- 83个因子按当前行情、收益、成交额和市值字段实现。
- 14个行业中性化因子以证监会一级/二级行业作代理，状态标记为 `PROXY_CSRC`。
- Alpha48、67、90、100需要历史细分行业，保留为空并标明原因。
- 所有时间窗口按论文规则对小数向下取整；所有 `rank` 都是同一交易日股票间的截面排名。
- 默认执行2016-01-01至2020-12-31、60只股票的真实数据烟雾测试；较长区间用于覆盖嵌套滚动窗口，完整研究应改用点时点股票池。

## Context & Methods

### Key Assumptions

1. `returns` 使用RESSET `日收益率_Dret`。
2. `volume` 直接使用股数，不再乘100；`advN` 使用成交金额（元）的N日均值。
3. `vwap = 成交金额(元) / 成交量(股)`。
4. `cap = 未复权收盘价 x 总股数`。
5. 默认价格模式使用 `Mcfacpr` 对OHLC和VWAP作同尺度前复权；可切换为原始价格。
6. 停牌行情保持缺失；只允许行业、股本和复权因子按股票向前填充，绝不向后填充未来数据。
7. 本Notebook计算因子值，不包含交易执行、涨跌停、费用和组合构建。

In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)
UNAVAILABLE_REASONS = {48: '需要逐日历史 IndClass.subindustry；当前RESSET文件只有证监会一级和二级行业，不能严格完成细分行业中性化。', 67: '同时需要 sector 与 subindustry；可代理sector，但缺少历史subindustry，因此保留空实现。', 90: '需要用历史subindustry对ADV40做截面中性化；当前数据缺少该层级。', 100: '公式两次使用subindustry中性化；当前数据缺少历史细分行业字段，不能严格实现。'}
PROXY_FACTORS = {97, 69, 70, 59, 76, 79, 80, 82, 87, 89, 58, 91, 93, 63}
ALPHA_METADATA = {1: {'formula': '(rank(Ts_ArgMax(SignedPower(((returns < 0) ? stddev(returns, 20) : close), 2.), 5)) -0.5)', 'definition': '当收益率为负时用20日收益波动率替代收盘价，否则使用收盘价；平方后寻找近5日最大值出现位置，再做当日截面排名并减0.5。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 2: {'formula': '(-1 * correlation(rank(delta(log(volume), 2)), rank(((close - open) / open)), 6))', 'definition': '衡量近6日内，对数成交量的2日变化与开盘到收盘日内收益之间的排名相关性，并取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 3: {'formula': '(-1 * correlation(rank(open), rank(volume), 10))', 'definition': '计算开盘价截面排名与成交量截面排名在近10日的相关性，并取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 4: {'formula': '(-1 * Ts_Rank(rank(low), 9))', 'definition': '先对最低价做每日截面排名，再计算其近9日时序排名并取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 5: {'formula': '(rank((open - (sum(vwap, 10) / 10))) * (-1 * abs(rank((close - vwap)))))', 'definition': '把开盘价相对10日VWAP均值的偏离排名，与收盘价相对VWAP偏离排名的负绝对值相乘。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 6: {'formula': '(-1 * correlation(open, volume, 10))', 'definition': '计算开盘价与成交量在近10日的时序相关性并取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 7: {'formula': '((adv20 < volume) ? ((-1 * ts_rank(abs(delta(close, 7)), 60)) * sign(delta(close, 7))) : (-1* 1))', 'definition': '仅在当日成交量高于20日平均成交额代理条件时，用7日价格变化的方向乘以其60日时序排名；否则取-1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 8: {'formula': '(-1 * rank(((sum(open, 5) * sum(returns, 5)) - delay((sum(open, 5) * sum(returns, 5)),10))))', 'definition': '将5日开盘价总和与5日收益率总和的乘积，相对10日前的变化做截面排名并取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 9: {'formula': '((0 < ts_min(delta(close, 1), 5)) ? delta(close, 1) : ((ts_max(delta(close, 1), 5) < 0) ?delta(close, 1) : (-1 * delta(close, 1))))', 'definition': '若近5日收盘价日变化始终同号，则保留当日变化；否则反转该变化。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 10: {'formula': 'rank(((0 < ts_min(delta(close, 1), 4)) ? delta(close, 1) : ((ts_max(delta(close, 1), 4) < 0)? delta(close, 1) : (-1 * delta(close, 1)))))', 'definition': '与Alpha9类似，但使用4日窗口，并对条件结果做每日截面排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 11: {'formula': '((rank(ts_max((vwap - close), 3)) + rank(ts_min((vwap - close), 3))) *rank(delta(volume, 3)))', 'definition': '将VWAP减收盘价的3日最大值排名和3日最小值排名相加，再乘以成交量3日变化排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 12: {'formula': '(sign(delta(volume, 1)) * (-1 * delta(close, 1)))', 'definition': '成交量日变化的符号，乘以收盘价日变化的相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 13: {'formula': '(-1 * rank(covariance(rank(close), rank(volume), 5)))', 'definition': '计算收盘价排名与成交量排名的5日协方差，对其做截面排名后取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 14: {'formula': '((-1 * rank(delta(returns, 3))) * correlation(open, volume, 10))', 'definition': '3日收益率变化的负截面排名，乘以开盘价与成交量的10日相关性。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 15: {'formula': '(-1 * sum(rank(correlation(rank(high), rank(volume), 3)), 3))', 'definition': '计算最高价排名和成交量排名的3日相关性，对相关性排名后求3日和并取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 16: {'formula': '(-1 * rank(covariance(rank(high), rank(volume), 5)))', 'definition': '计算最高价排名与成交量排名的5日协方差，做截面排名后取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 17: {'formula': '(((-1 * rank(ts_rank(close, 10))) * rank(delta(delta(close, 1), 1))) *rank(ts_rank((volume / adv20), 5)))', 'definition': '把收盘价10日时序排名、价格变化的二阶差分以及量能相对ADV20的5日时序排名组合为负向乘积。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 18: {'formula': '(-1 * rank(((stddev(abs((close - open)), 5) + (close - open)) + correlation(close, open,10))))', 'definition': '把开收盘价差的5日波动、当日开收盘价差及开盘价与收盘价10日相关性相加，排名后取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 19: {'formula': '((-1 * sign(((close - delay(close, 7)) + delta(close, 7)))) * (1 + rank((1 + sum(returns,250)))))', 'definition': '以7日价格变化方向构造反转信号，并用250日累计收益的截面排名调整强度。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 20: {'formula': '(((-1 * rank((open - delay(high, 1)))) * rank((open - delay(close, 1)))) * rank((open -delay(low, 1))))', 'definition': '比较今日开盘价与昨日最高、收盘、最低价的差异，分别排名后相乘并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 21: {'formula': '((((sum(close, 8) / 8) + stddev(close, 8)) < (sum(close, 2) / 2)) ? (-1 * 1) : (((sum(close,2) / 2) < ((sum(close, 8) / 8) - stddev(close, 8))) ? 1 : (((1 < (volume / adv20)) || ((volume /adv20) == 1)) ? 1 : (-1 * 1))))', 'definition': '根据短期均价相对8日均值加减波动带的位置给出-1或1；若不触发价格条件，再由量能相对ADV20决定。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 22: {'formula': '(-1 * (delta(correlation(high, volume, 5), 5) * rank(stddev(close, 20))))', 'definition': '最高价与成交量5日相关性的5日变化，乘以收盘价20日波动率排名并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 23: {'formula': '(((sum(high, 20) / 20) < high) ? (-1 * delta(high, 2)) : 0)', 'definition': '当最高价高于其20日均值时，取最高价2日变化的相反数，否则为0。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 24: {'formula': '((((delta((sum(close, 100) / 100), 100) / delay(close, 100)) < 0.05) ||((delta((sum(close, 100) / 100), 100) / delay(close, 100)) == 0.05)) ? (-1 * (close - ts_min(close,100))) : (-1 * delta(close, 3)))', 'definition': '当100日均价趋势相对100日前收盘价不超过5%时使用距100日最低价的负偏离，否则使用3日价格变化的相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 25: {'formula': 'rank(((((-1 * returns) * adv20) * vwap) * (high - close)))', 'definition': '将负收益、ADV20、VWAP以及最高价减收盘价的乘积做每日截面排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 26: {'formula': '(-1 * ts_max(correlation(ts_rank(volume, 5), ts_rank(high, 5), 5), 3))', 'definition': '成交量5日时序排名与最高价5日时序排名的5日相关性，取其近3日最大值并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 27: {'formula': '((0.5 < rank((sum(correlation(rank(volume), rank(vwap), 6), 2) / 2.0))) ? (-1 * 1) : 1)', 'definition': '将成交量排名和VWAP排名的6日相关性做2日平均及截面排名，高于0.5取-1，否则取1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 28: {'formula': 'scale(((correlation(adv20, low, 5) + ((high + low) / 2)) - close))', 'definition': '把ADV20与最低价的5日相关性、日内中间价和收盘价组合后，按日做绝对值和为1的截面缩放。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 29: {'formula': '(min(product(rank(rank(scale(log(sum(ts_min(rank(rank((-1 * rank(delta((close - 1),5))))), 2), 1))))), 1), 5) + ts_rank(delay((-1 * returns), 6), 5))', 'definition': '对价格变化排名进行多层排名、缩放、对数和短窗极值处理，再加上滞后负收益的5日时序排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 30: {'formula': '(((1.0 - rank(((sign((close - delay(close, 1))) + sign((delay(close, 1) - delay(close, 2)))) +sign((delay(close, 2) - delay(close, 3)))))) * sum(volume, 5)) / sum(volume, 20))', 'definition': '用最近三次价格变化方向之和的截面排名构造反转项，再乘以5日成交量总和相对20日总和的比例。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 31: {'formula': '((rank(rank(rank(decay_linear((-1 * rank(rank(delta(close, 10)))), 10)))) + rank((-1 *delta(close, 3)))) + sign(scale(correlation(adv20, low, 12))))', 'definition': '组合10日价格变化的线性衰减多重排名、3日价格变化负排名，以及ADV20和最低价相关性的方向。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 32: {'formula': '(scale(((sum(close, 7) / 7) - close)) + (20 * scale(correlation(vwap, delay(close, 5),230))))', 'definition': '组合收盘价相对7日均价的截面缩放项，以及VWAP与滞后5日收盘价的230日相关性缩放项。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 33: {'formula': 'rank((-1 * ((1 - (open / close))^1)))', 'definition': '对开盘价相对收盘价的比例减1做每日截面排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 34: {'formula': 'rank(((1 - rank((stddev(returns, 2) / stddev(returns, 5)))) + (1 - rank(delta(close, 1)))))', 'definition': '结合2日与5日收益波动率之比的反排名，以及收盘价日变化的反排名，再做截面排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 35: {'formula': '((Ts_Rank(volume, 32) * (1 - Ts_Rank(((close + high) - low), 16))) * (1 -Ts_Rank(returns, 32)))', 'definition': '将成交量32日时序排名、价格区间组合的16日反时序排名和收益率32日反时序排名相乘。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 36: {'formula': '(((((2.21 * rank(correlation((close - open), delay(volume, 1), 15))) + (0.7 * rank((open- close)))) + (0.73 * rank(Ts_Rank(delay((-1 * returns), 6), 5)))) + rank(abs(correlation(vwap,adv20, 6)))) + (0.6 * rank((((sum(close, 200) / 200) - open) * (close - open)))))', 'definition': '按给定权重组合开收盘差与滞后成交量相关性、开收盘差、滞后负收益排名、VWAP与ADV20相关性及长期均价偏离。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 37: {'formula': '(rank(correlation(delay((open - close), 1), close, 200)) + rank((open - close)))', 'definition': '把昨日开收盘差与收盘价的200日相关性排名，加上当日开收盘差排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 38: {'formula': '((-1 * rank(Ts_Rank(close, 10))) * rank((close / open)))', 'definition': '收盘价10日时序排名的负截面排名，乘以收盘价相对开盘价比例的截面排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 39: {'formula': '((-1 * rank((delta(close, 7) * (1 - rank(decay_linear((volume / adv20), 9)))))) * (1 +rank(sum(returns, 250))))', 'definition': '以7日价格变化和量能相对ADV20的线性衰减排名构造反向项，再由250日累计收益排名放大。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 40: {'formula': '((-1 * rank(stddev(high, 10))) * correlation(high, volume, 10))', 'definition': '最高价10日波动率的负截面排名，乘以最高价与成交量的10日相关性。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 41: {'formula': '(((high * low)^0.5) - vwap)', 'definition': '最高价与最低价几何平均值减去VWAP。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 42: {'formula': '(rank((vwap - close)) / rank((vwap + close)))', 'definition': 'VWAP减收盘价的截面排名，除以VWAP加收盘价的截面排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 43: {'formula': '(ts_rank((volume / adv20), 20) * ts_rank((-1 * delta(close, 7)), 8))', 'definition': '量能相对ADV20的20日时序排名，乘以负7日价格变化的8日时序排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 44: {'formula': '(-1 * correlation(high, rank(volume), 5))', 'definition': '最高价与成交量截面排名的5日相关性，并取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 45: {'formula': '(-1 * ((rank((sum(delay(close, 5), 20) / 20)) * correlation(close, volume, 2)) *rank(correlation(sum(close, 5), sum(close, 20), 2))))', 'definition': '组合滞后收盘价20日均值排名、收盘价与成交量2日相关性，以及5日与20日价格总和相关性的排名，并整体取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 46: {'formula': '((0.25 < (((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10))) ?(-1 * 1) : (((((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10)) < 0) ? 1 :((-1 * 1) * (close - delay(close, 1)))))', 'definition': '根据10日与20日价格趋势差判断：高于0.25取-1，低于0取1，否则取当日价格变化的相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 47: {'formula': '((((rank((1 / close)) * volume) / adv20) * ((high * rank((high - close))) / (sum(high, 5) /5))) - rank((vwap - delay(vwap, 5))))', 'definition': '结合低价股排名、成交量相对ADV20、最高价位置，并减去VWAP相对5日前变化的排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 48: {'formula': '(indneutralize(((correlation(delta(close, 1), delta(delay(close, 1), 1), 250) *delta(close, 1)) / close), IndClass.subindustry) / sum(((delta(close, 1) / delay(close, 1))^2), 250))', 'definition': '将价格变化自相关相关项按细分行业中性化，再除以250日收益变化平方和。', 'status': 'UNAVAILABLE', 'reason': '需要逐日历史 IndClass.subindustry；当前RESSET文件只有证监会一级和二级行业，不能严格完成细分行业中性化。'}, 49: {'formula': '(((((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10)) < (-1 *0.1)) ? 1 : ((-1 * 1) * (close - delay(close, 1))))', 'definition': '当10日与20日价格趋势差低于-0.1时取1，否则取当日价格变化的相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 50: {'formula': '(-1 * ts_max(rank(correlation(rank(volume), rank(vwap), 5)), 5))', 'definition': '成交量排名与VWAP排名5日相关性的截面排名，取近5日最大值并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 51: {'formula': '(((((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10)) < (-1 *0.05)) ? 1 : ((-1 * 1) * (close - delay(close, 1))))', 'definition': '与Alpha49相同，但趋势触发阈值改为-0.05。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 52: {'formula': '((((-1 * ts_min(low, 5)) + delay(ts_min(low, 5), 5)) * rank(((sum(returns, 240) -sum(returns, 20)) / 220))) * ts_rank(volume, 5))', 'definition': '结合5日最低价的5日反向变化、240日与20日累计收益差的排名，以及成交量5日时序排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 53: {'formula': '(-1 * delta((((close - low) - (high - close)) / (close - low)), 9))', 'definition': '对K线收盘位置比率做9日变化并取相反数。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 54: {'formula': '((-1 * ((low - close) * (open^5))) / ((low - high) * (close^5)))', 'definition': '用开盘价和收盘价的五次幂，结合最低价与收盘价、最高价的距离构造非线性日内信号。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 55: {'formula': '(-1 * correlation(rank(((close - ts_min(low, 12)) / (ts_max(high, 12) - ts_min(low,12)))), rank(volume), 6))', 'definition': '收盘价在12日高低区间中的位置排名，与成交量排名的6日相关性取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 56: {'formula': '(0 - (1 * (rank((sum(returns, 10) / sum(sum(returns, 2), 3))) * rank((returns * cap)))))', 'definition': '10日收益总和相对嵌套短期收益总和的排名，乘以收益率与总市值乘积的排名并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 57: {'formula': '(0 - (1 * ((close - vwap) / decay_linear(rank(ts_argmax(close, 30)), 2))))', 'definition': '收盘价减VWAP，除以收盘价30日最高值出现位置排名的2日线性衰减值，并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 58: {'formula': '(-1 * Ts_Rank(decay_linear(correlation(IndNeutralize(vwap, IndClass.sector), volume,3.92795), 7.89291), 5.50322))', 'definition': '对行业板块中性化后的VWAP与成交量做短期相关、线性衰减及时序排名，并取负；本适配版以证监会一级行业代理sector。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 59: {'formula': '(-1 * Ts_Rank(decay_linear(correlation(IndNeutralize(((vwap * 0.728317) + (vwap *(1 - 0.728317))), IndClass.industry), volume, 4.25197), 16.2289), 8.19648))', 'definition': '对行业中性化VWAP与成交量的短期相关性做线性衰减和时序排名并取负；本适配版以证监会二级行业代理industry。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 60: {'formula': '(0 - (1 * ((2 * scale(rank(((((close - low) - (high - close)) / (high - low)) * volume)))) -scale(rank(ts_argmax(close, 10))))))', 'definition': '结合收盘价在日内区间的位置乘成交量的排名，以及收盘价10日最高值出现位置排名，分别缩放后相减并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 61: {'formula': '(rank((vwap - ts_min(vwap, 16.1219))) < rank(correlation(vwap, adv180, 17.9282)))', 'definition': '比较VWAP距近16日最低值的排名，与VWAP和ADV180近17日相关性的排名，输出0或1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 62: {'formula': '((rank(correlation(vwap, sum(adv20, 22.4101), 9.91009)) < rank(((rank(open) +rank(open)) < (rank(((high + low) / 2)) + rank(high))))) * -1)', 'definition': '比较VWAP与ADV20滚动总和相关性的排名，和开盘价排名相对高低价排名组合的布尔排名，条件成立时取-1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 63: {'formula': '((rank(decay_linear(delta(IndNeutralize(close, IndClass.industry), 2.25164), 8.22237))- rank(decay_linear(correlation(((vwap * 0.318108) + (open * (1 - 0.318108))), sum(adv180,37.2467), 13.557), 12.2883))) * -1)', 'definition': '比较行业中性化收盘价变化的衰减排名，与VWAP/开盘混合价同ADV180总和相关性的衰减排名；以证监会二级行业代理。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 64: {'formula': '((rank(correlation(sum(((open * 0.178404) + (low * (1 - 0.178404))), 12.7054),sum(adv120, 12.7054), 16.6208)) < rank(delta(((((high + low) / 2) * 0.178404) + (vwap * (1 -0.178404))), 3.69741))) * -1)', 'definition': '比较开盘价/最低价混合价总和与ADV120总和的相关性排名，以及中间价/VWAP混合价变化的排名，成立时取-1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 65: {'formula': '((rank(correlation(((open * 0.00817205) + (vwap * (1 - 0.00817205))), sum(adv60,8.6911), 6.40374)) < rank((open - ts_min(open, 13.635)))) * -1)', 'definition': '比较开盘价/VWAP混合价与ADV60总和相关性的排名，以及开盘价距近13日最低值的排名，成立时取-1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 66: {'formula': '((rank(decay_linear(delta(vwap, 3.51013), 7.23052)) + Ts_Rank(decay_linear(((((low* 0.96633) + (low * (1 - 0.96633))) - vwap) / (open - ((high + low) / 2))), 11.4157), 6.72611)) * -1)', 'definition': '把VWAP变化的衰减排名，与最低价相对VWAP和开盘价相对中间价之比的衰减时序排名相加并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 67: {'formula': '((rank((high - ts_min(high, 2.14593)))^rank(correlation(IndNeutralize(vwap,IndClass.sector), IndNeutralize(adv20, IndClass.subindustry), 6.02936))) * -1)', 'definition': '将最高价距短期最低值的排名，与sector中性化VWAP和subindustry中性化ADV20相关性的排名作幂运算并取负。', 'status': 'UNAVAILABLE', 'reason': '同时需要 sector 与 subindustry；可代理sector，但缺少历史subindustry，因此保留空实现。'}, 68: {'formula': '((Ts_Rank(correlation(rank(high), rank(adv15), 8.91644), 13.9333) <rank(delta(((close * 0.518371) + (low * (1 - 0.518371))), 1.06157))) * -1)', 'definition': '比较最高价排名与ADV15排名相关性的时序排名，以及收盘/最低混合价变化的排名，成立时取-1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 69: {'formula': '((rank(ts_max(delta(IndNeutralize(vwap, IndClass.industry), 2.72412),4.79344))^Ts_Rank(correlation(((close * 0.490655) + (vwap * (1 - 0.490655))), adv20, 4.92416),9.0615)) * -1)', 'definition': '行业中性化VWAP变化的短期最大值排名，按价格混合项与ADV20相关性的时序排名做幂运算并取负。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 70: {'formula': '((rank(delta(vwap, 1.29456))^Ts_Rank(correlation(IndNeutralize(close,IndClass.industry), adv50, 17.8256), 17.9171)) * -1)', 'definition': 'VWAP变化排名，按行业中性化收盘价与ADV50相关性的时序排名做幂运算并取负。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 71: {'formula': 'max(Ts_Rank(decay_linear(correlation(Ts_Rank(close, 3.43976), Ts_Rank(adv180,12.0647), 18.0175), 4.20501), 15.6948), Ts_Rank(decay_linear((rank(((low + open) - (vwap +vwap)))^2), 16.4662), 4.4388))', 'definition': '取两项较大者：价格时序排名与ADV180时序排名相关性的衰减时序排名；价格位置排名平方的衰减时序排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 72: {'formula': '(rank(decay_linear(correlation(((high + low) / 2), adv40, 8.93345), 10.1519)) /rank(decay_linear(correlation(Ts_Rank(vwap, 3.72469), Ts_Rank(volume, 18.5188), 6.86671),2.95011)))', 'definition': '中间价与ADV40相关性的衰减排名，除以VWAP时序排名与成交量时序排名相关性的衰减排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 73: {'formula': '(max(rank(decay_linear(delta(vwap, 4.72775), 2.91864)),Ts_Rank(decay_linear(((delta(((open * 0.147155) + (low * (1 - 0.147155))), 2.03608) / ((open *0.147155) + (low * (1 - 0.147155)))) * -1), 3.33829), 16.7411)) * -1)', 'definition': '取VWAP变化衰减排名与开盘/最低混合价相对变化衰减时序排名中的较大者，并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 74: {'formula': '((rank(correlation(close, sum(adv30, 37.4843), 15.1365)) <rank(correlation(rank(((high * 0.0261661) + (vwap * (1 - 0.0261661)))), rank(volume), 11.4791)))* -1)', 'definition': '比较收盘价与ADV30总和相关性的排名，以及最高价/VWAP混合价排名与成交量排名相关性的排名，成立时取-1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 75: {'formula': '(rank(correlation(vwap, volume, 4.24304)) < rank(correlation(rank(low), rank(adv50),12.4413)))', 'definition': '比较VWAP与成交量相关性的排名，以及最低价排名与ADV50排名相关性的排名，输出0或1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 76: {'formula': '(max(rank(decay_linear(delta(vwap, 1.24383), 11.8259)),Ts_Rank(decay_linear(Ts_Rank(correlation(IndNeutralize(low, IndClass.sector), adv81,8.14941), 19.569), 17.1543), 19.383)) * -1)', 'definition': '取VWAP变化衰减排名与sector中性化最低价同ADV81相关项的双重时序处理中的较大者并取负；sector用证监会一级行业代理。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 77: {'formula': 'min(rank(decay_linear(((((high + low) / 2) + high) - (vwap + high)), 20.0451)),rank(decay_linear(correlation(((high + low) / 2), adv40, 3.1614), 5.64125)))', 'definition': '取中间价相对VWAP偏离的衰减排名，与中间价和ADV40相关性的衰减排名中的较小者。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 78: {'formula': '(rank(correlation(sum(((low * 0.352233) + (vwap * (1 - 0.352233))), 19.7428),sum(adv40, 19.7428), 6.83313))^rank(correlation(rank(vwap), rank(volume), 5.77492)))', 'definition': '价格混合项总和与ADV40总和相关性的排名，按VWAP排名与成交量排名相关性的排名做幂运算。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 79: {'formula': '(rank(delta(IndNeutralize(((close * 0.60733) + (open * (1 - 0.60733))),IndClass.sector), 1.23438)) < rank(correlation(Ts_Rank(vwap, 3.60973), Ts_Rank(adv150,9.18637), 14.6644)))', 'definition': '比较sector中性化收盘/开盘混合价变化的排名，以及VWAP和ADV150时序排名相关性的排名；sector用证监会一级行业代理。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 80: {'formula': '((rank(Sign(delta(IndNeutralize(((open * 0.868128) + (high * (1 - 0.868128))),IndClass.industry), 4.04545)))^Ts_Rank(correlation(high, adv10, 5.11456), 5.53756)) * -1)', 'definition': '行业中性化开盘/最高混合价变化方向的排名，按最高价与ADV10相关性的时序排名做幂运算并取负。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 81: {'formula': '((rank(Log(product(rank((rank(correlation(vwap, sum(adv10, 49.6054),8.47743))^4)), 14.9655))) < rank(correlation(rank(vwap), rank(volume), 5.07914))) * -1)', 'definition': '比较VWAP与ADV10总和相关项经幂、排名、连乘和对数后的排名，以及VWAP排名与成交量排名相关性的排名，成立时取-1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 82: {'formula': '(min(rank(decay_linear(delta(open, 1.46063), 14.8717)),Ts_Rank(decay_linear(correlation(IndNeutralize(volume, IndClass.sector), ((open * 0.634196) +(open * (1 - 0.634196))), 17.4842), 6.92131), 13.4283)) * -1)', 'definition': '取开盘价变化衰减排名和sector中性化成交量与开盘价相关项的衰减时序排名中的较小者并取负。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 83: {'formula': '((rank(delay(((high - low) / (sum(close, 5) / 5)), 2)) * rank(rank(volume))) / (((high -low) / (sum(close, 5) / 5)) / (vwap - close)))', 'definition': '把滞后的日内振幅相对5日均价之比的排名与成交量双重排名相乘，再除以当日振幅相对均价及VWAP收盘偏离的组合。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 84: {'formula': 'SignedPower(Ts_Rank((vwap - ts_max(vwap, 15.3217)), 20.7127), delta(close,4.96796))', 'definition': 'VWAP距近15日最高值的20日时序排名，以收盘价4日变化为指数做有符号幂运算。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 85: {'formula': '(rank(correlation(((high * 0.876703) + (close * (1 - 0.876703))), adv30,9.61331))^rank(correlation(Ts_Rank(((high + low) / 2), 3.70596), Ts_Rank(volume, 10.1595),7.11408)))', 'definition': '最高/收盘混合价与ADV30相关性的排名，按中间价时序排名与成交量时序排名相关性的排名做幂运算。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 86: {'formula': '((Ts_Rank(correlation(close, sum(adv20, 14.7444), 6.00049), 20.4195) < rank(((open+ close) - (vwap + open)))) * -1)', 'definition': '比较收盘价与ADV20总和相关性的时序排名，以及收盘价减VWAP的截面排名，成立时取-1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 87: {'formula': '(max(rank(decay_linear(delta(((close * 0.369701) + (vwap * (1 - 0.369701))),1.91233), 2.65461)), Ts_Rank(decay_linear(abs(correlation(IndNeutralize(adv81,IndClass.industry), close, 13.4132)), 4.89768), 14.4535)) * -1)', 'definition': '取价格混合项变化的衰减排名，与行业中性化ADV81同收盘价相关性绝对值的衰减时序排名中的较大者并取负。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 88: {'formula': 'min(rank(decay_linear(((rank(open) + rank(low)) - (rank(high) + rank(close))),8.06882)), Ts_Rank(decay_linear(correlation(Ts_Rank(close, 8.44728), Ts_Rank(adv60,20.6966), 8.01266), 6.65053), 2.61957))', 'definition': '取四种价格排名差的衰减排名，与收盘价和ADV60各自时序排名相关性的衰减时序排名中的较小者。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 89: {'formula': '(Ts_Rank(decay_linear(correlation(((low * 0.967285) + (low * (1 - 0.967285))), adv10,6.94279), 5.51607), 3.79744) - Ts_Rank(decay_linear(delta(IndNeutralize(vwap,IndClass.industry), 3.48158), 10.1466), 15.3012))', 'definition': '最低价与ADV10相关性的衰减时序排名，减去行业中性化VWAP变化的衰减时序排名。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 90: {'formula': '((rank((close - ts_max(close, 4.66719)))^Ts_Rank(correlation(IndNeutralize(adv40,IndClass.subindustry), low, 5.38375), 3.21856)) * -1)', 'definition': '收盘价距短期最高值的排名，按subindustry中性化ADV40与最低价相关性的时序排名做幂运算并取负。', 'status': 'UNAVAILABLE', 'reason': '需要用历史subindustry对ADV40做截面中性化；当前数据缺少该层级。'}, 91: {'formula': '((Ts_Rank(decay_linear(decay_linear(correlation(IndNeutralize(close,IndClass.industry), volume, 9.74928), 16.398), 3.83219), 4.8667) -rank(decay_linear(correlation(vwap, adv30, 4.01303), 2.6809))) * -1)', 'definition': '行业中性化收盘价与成交量相关项经两次衰减后的时序排名，减去VWAP与ADV30相关性的衰减排名，整体取负。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 92: {'formula': 'min(Ts_Rank(decay_linear(((((high + low) / 2) + close) < (low + open)), 14.7221),18.8683), Ts_Rank(decay_linear(correlation(rank(low), rank(adv30), 7.58555), 6.94024),6.80584))', 'definition': '取价格不等式布尔值的衰减时序排名，与最低价排名和ADV30排名相关性的衰减时序排名中的较小者。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 93: {'formula': '(Ts_Rank(decay_linear(correlation(IndNeutralize(vwap, IndClass.industry), adv81,17.4193), 19.848), 7.54455) / rank(decay_linear(delta(((close * 0.524434) + (vwap * (1 -0.524434))), 2.77377), 16.2664)))', 'definition': '行业中性化VWAP与ADV81相关性的衰减时序排名，除以收盘/VWAP混合价变化的衰减排名。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 94: {'formula': '((rank((vwap - ts_min(vwap, 11.5783)))^Ts_Rank(correlation(Ts_Rank(vwap,19.6462), Ts_Rank(adv60, 4.02992), 18.0926), 2.70756)) * -1)', 'definition': 'VWAP距近11日最低值的排名，按VWAP与ADV60时序排名相关性的时序排名做幂运算并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 95: {'formula': '(rank((open - ts_min(open, 12.4105))) < Ts_Rank((rank(correlation(sum(((high + low)/ 2), 19.1351), sum(adv40, 19.1351), 12.8742))^5), 11.7584))', 'definition': '比较开盘价距短期最低值的排名，以及中间价总和与ADV40总和相关性排名五次幂的时序排名，输出0或1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 96: {'formula': '(max(Ts_Rank(decay_linear(correlation(rank(vwap), rank(volume), 3.83878),4.16783), 8.38151), Ts_Rank(decay_linear(Ts_ArgMax(correlation(Ts_Rank(close, 7.45404),Ts_Rank(adv60, 4.13242), 3.65459), 12.6556), 14.0365), 13.4143)) * -1)', 'definition': '取VWAP排名与成交量排名相关性的衰减时序排名，和价格/ADV60时序相关性最大值位置的衰减时序排名中的较大者并取负。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 97: {'formula': '((rank(decay_linear(delta(IndNeutralize(((low * 0.721001) + (vwap * (1 - 0.721001))),IndClass.industry), 3.3705), 20.4523)) - Ts_Rank(decay_linear(Ts_Rank(correlation(Ts_Rank(low,7.87871), Ts_Rank(adv60, 17.255), 4.97547), 18.5925), 15.7152), 6.71659)) * -1)', 'definition': '比较行业中性化最低价/VWAP混合价变化的衰减排名，与最低价和ADV60多层时序相关项的衰减时序排名，整体取负。', 'status': 'PROXY_CSRC', 'reason': '公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。'}, 98: {'formula': '(rank(decay_linear(correlation(vwap, sum(adv5, 26.4719), 4.58418), 7.18088)) -rank(decay_linear(Ts_Rank(Ts_ArgMin(correlation(rank(open), rank(adv15), 20.8187), 8.62571),6.95668), 8.07206)))', 'definition': 'VWAP与ADV5总和相关性的衰减排名，减去开盘价排名与ADV15排名相关性最小值位置经时序排名和衰减后的排名。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 99: {'formula': '((rank(correlation(sum(((high + low) / 2), 19.8975), sum(adv60, 19.8975), 8.8136)) <rank(correlation(low, volume, 6.28259))) * -1)', 'definition': '比较中间价总和与ADV60总和相关性的排名，以及最低价与成交量相关性的排名，成立时取-1。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}, 100: {'formula': '(0 - (1 * (((1.5 * scale(indneutralize(indneutralize(rank(((((close - low) - (high -close)) / (high - low)) * volume)), IndClass.subindustry), IndClass.subindustry))) -scale(indneutralize((correlation(close, rank(adv20), 5) - rank(ts_argmin(close, 30))),IndClass.subindustry))) * (volume / adv20))))', 'definition': '将量价日内位置项两次按subindustry中性化并缩放，再减去另一subindustry中性化相关项的缩放值，最后乘量能相对ADV20并取负。', 'status': 'UNAVAILABLE', 'reason': '公式两次使用subindustry中性化；当前数据缺少历史细分行业字段，不能严格实现。'}, 101: {'formula': '((close - open) / ((high - low) + .001))', 'definition': '收盘价减开盘价，除以当日最高价减最低价加0.001，衡量收盘在日内价格区间中的方向与幅度。', 'status': 'IMPLEMENTED', 'reason': '当前RESSET字段可支持。'}}
metadata_table = pd.DataFrame.from_dict(ALPHA_METADATA, orient='index')
metadata_table.index.name = 'alpha_number'
metadata_table.groupby('status').size().rename('count').to_frame()

,count
status,
IMPLEMENTED,83
PROXY_CSRC,14
UNAVAILABLE,4


## Data

数据源：`D:\因子分析\数据\RESSET_DRESSTK_2016_2025.parquet`。

下面的加载逻辑使用实际交易日历、去重后的股票-日期主键，并把停牌日行情保留为缺失值。

In [2]:

from __future__ import annotations

import math
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


def _window(value) -> int:
    """论文规定：非整数窗口向下取整。"""
    return max(1, int(math.floor(float(value))))


def _as_float(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.astype(float) if all(dtype == bool for dtype in frame.dtypes) else frame


def _clean(frame: pd.DataFrame) -> pd.DataFrame:
    return _as_float(frame).replace([np.inf, -np.inf], np.nan)


def rank(frame):
    """每日股票间截面百分位排名，而不是沿单只股票时间轴排名。"""
    return frame.rank(axis=1, pct=True, method="average")


def delay(frame, days=1):
    return frame.shift(_window(days))


def delta(frame, days=1):
    return frame - delay(frame, days)


def ts_sum(frame, days):
    days = _window(days)
    return frame.rolling(days, min_periods=days).sum()


def ts_mean(frame, days):
    days = _window(days)
    return frame.rolling(days, min_periods=days).mean()


def stddev(frame, days):
    days = _window(days)
    return frame.rolling(days, min_periods=days).std(ddof=1)


def correlation(left, right, days):
    days = _window(days)
    result = left.rolling(days, min_periods=days).corr(right)
    valid_pairs = (left.notna() & right.notna()).rolling(days, min_periods=days).sum()
    # 完整窗口内若一侧为常数，相关系数数学上未定义；按0相关处理，而不是向前/向后填充。
    return result.mask(result.isna() & valid_pairs.eq(days), 0.0)


def covariance(left, right, days):
    days = _window(days)
    return left.rolling(days, min_periods=days).cov(right)


def ts_min(frame, days):
    days = _window(days)
    return frame.rolling(days, min_periods=days).min()


def ts_max(frame, days):
    days = _window(days)
    return frame.rolling(days, min_periods=days).max()


def ts_product(frame, days):
    days = _window(days)
    return frame.rolling(days, min_periods=days).apply(np.prod, raw=True)


def _last_pct_rank(values):
    last = values[-1]
    less = np.sum(values < last)
    equal = np.sum(values == last)
    return (less + (equal + 1.0) / 2.0) / len(values)


def ts_rank(frame, days):
    days = _window(days)
    return frame.rolling(days, min_periods=days).apply(_last_pct_rank, raw=True)


def ts_argmax(frame, days):
    days = _window(days)
    return frame.rolling(days, min_periods=days).apply(lambda x: float(np.argmax(x) + 1), raw=True)


def ts_argmin(frame, days):
    days = _window(days)
    return frame.rolling(days, min_periods=days).apply(lambda x: float(np.argmin(x) + 1), raw=True)


def decay_linear(frame, days):
    days = _window(days)
    weights = np.arange(1.0, days + 1.0)
    weights /= weights.sum()
    numeric = frame.astype(float)
    return numeric.rolling(days, min_periods=days).apply(lambda x: float(np.dot(x, weights)), raw=True)


def scale(frame, amount=1.0):
    denominator = frame.abs().sum(axis=1).replace(0, np.nan)
    return frame.div(denominator, axis=0) * float(amount)


def signed_power(frame, exponent):
    return np.sign(frame) * np.power(np.abs(frame), exponent)


def safe_divide(numerator, denominator):
    if isinstance(denominator, pd.DataFrame):
        denominator = denominator.where(denominator.abs() > 1e-12)
    return numerator / denominator


def where(condition, when_true, when_false):
    index, columns = condition.index, condition.columns
    true_values = when_true.to_numpy() if isinstance(when_true, pd.DataFrame) else when_true
    false_values = when_false.to_numpy() if isinstance(when_false, pd.DataFrame) else when_false
    return pd.DataFrame(np.where(condition.to_numpy(), true_values, false_values), index=index, columns=columns)


def minimum(left, right):
    return left.where(left <= right, right)


def maximum(left, right):
    return left.where(left >= right, right)


def neutralize(frame, groups):
    """逐日、组内去均值。groups必须是同形状的历史点时点分类面板。"""
    result = pd.DataFrame(np.nan, index=frame.index, columns=frame.columns, dtype=float)
    for date in frame.index:
        values = frame.loc[date]
        labels = groups.loc[date]
        valid = values.notna() & labels.notna()
        if not valid.any():
            continue
        valid_values = values.loc[valid]
        valid_labels = labels.loc[valid]
        group_mean = valid_values.groupby(valid_labels).transform("mean")
        result.loc[date, valid] = valid_values - group_mean
    return result


def trend_acceleration(close):
    return ((delay(close, 20) - delay(close, 10)) / 10.0) - ((delay(close, 10) - close) / 10.0)


PYTHON_EXPRESSIONS = {
    1: "rank(ts_argmax(signed_power(where(returns < 0, stddev(returns, 20), close), 2.0), 5)) - 0.5",
    2: "-correlation(rank(delta(log(volume), 2)), rank(safe_divide(close - open_, open_)), 6)",
    3: "-correlation(rank(open_), rank(volume), 10)",
    4: "-ts_rank(rank(low), 9)",
    5: "rank(open_ - ts_sum(vwap, 10) / 10.0) * (-abs_(rank(close - vwap)))",
    6: "-correlation(open_, volume, 10)",
    7: "where(adv(20) < volume, -ts_rank(abs_(delta(close, 7)), 60) * sign(delta(close, 7)), -1.0)",
    8: "-rank(delta(ts_sum(open_, 5) * ts_sum(returns, 5), 10))",
    9: "where(ts_min(delta(close, 1), 5) > 0, delta(close, 1), where(ts_max(delta(close, 1), 5) < 0, delta(close, 1), -delta(close, 1)))",
    10: "rank(where(ts_min(delta(close, 1), 4) > 0, delta(close, 1), where(ts_max(delta(close, 1), 4) < 0, delta(close, 1), -delta(close, 1))))",
    11: "(rank(ts_max(vwap - close, 3)) + rank(ts_min(vwap - close, 3))) * rank(delta(volume, 3))",
    12: "sign(delta(volume, 1)) * (-delta(close, 1))",
    13: "-rank(covariance(rank(close), rank(volume), 5))",
    14: "-rank(delta(returns, 3)) * correlation(open_, volume, 10)",
    15: "-ts_sum(rank(correlation(rank(high), rank(volume), 3)), 3)",
    16: "-rank(covariance(rank(high), rank(volume), 5))",
    17: "-rank(ts_rank(close, 10)) * rank(delta(delta(close, 1), 1)) * rank(ts_rank(safe_divide(volume, adv(20)), 5))",
    18: "-rank(stddev(abs_(close - open_), 5) + (close - open_) + correlation(close, open_, 10))",
    19: "-sign((close - delay(close, 7)) + delta(close, 7)) * (1 + rank(1 + ts_sum(returns, 250)))",
    20: "-rank(open_ - delay(high, 1)) * rank(open_ - delay(close, 1)) * rank(open_ - delay(low, 1))",
    21: "where(ts_mean(close, 8) + stddev(close, 8) < ts_mean(close, 2), -1.0, where(ts_mean(close, 2) < ts_mean(close, 8) - stddev(close, 8), 1.0, where(safe_divide(volume, adv(20)) >= 1, 1.0, -1.0)))",
    22: "-delta(correlation(high, volume, 5), 5) * rank(stddev(close, 20))",
    23: "where(ts_mean(high, 20) < high, -delta(high, 2), 0.0)",
    24: "where(safe_divide(delta(ts_mean(close, 100), 100), delay(close, 100)) <= 0.05, -(close - ts_min(close, 100)), -delta(close, 3))",
    25: "rank((-returns) * adv(20) * vwap * (high - close))",
    26: "-ts_max(correlation(ts_rank(volume, 5), ts_rank(high, 5), 5), 3)",
    27: "where(rank(ts_sum(correlation(rank(volume), rank(vwap), 6), 2) / 2.0) > 0.5, -1.0, 1.0)",
    28: "scale(correlation(adv(20), low, 5) + (high + low) / 2.0 - close)",
    29: "ts_min(ts_product(rank(rank(scale(log(ts_sum(ts_min(rank(rank(-rank(delta(close - 1, 5)))), 2), 1))))), 1), 5) + ts_rank(delay(-returns, 6), 5)",
    30: "safe_divide((1.0 - rank(sign(delta(close, 1)) + sign(delay(delta(close, 1), 1)) + sign(delay(delta(close, 1), 2)))) * ts_sum(volume, 5), ts_sum(volume, 20))",
    31: "rank(rank(rank(decay_linear(-rank(rank(delta(close, 10))), 10)))) + rank(-delta(close, 3)) + sign(scale(correlation(adv(20), low, 12)))",
    32: "scale(ts_sum(close, 7) / 7.0 - close) + 20.0 * scale(correlation(vwap, delay(close, 5), 230))",
    33: "rank(safe_divide(open_, close) - 1.0)",
    34: "rank((1.0 - rank(safe_divide(stddev(returns, 2), stddev(returns, 5)))) + (1.0 - rank(delta(close, 1))))",
    35: "ts_rank(volume, 32) * (1.0 - ts_rank(close + high - low, 16)) * (1.0 - ts_rank(returns, 32))",
    36: "2.21 * rank(correlation(close - open_, delay(volume, 1), 15)) + 0.7 * rank(open_ - close) + 0.73 * rank(ts_rank(delay(-returns, 6), 5)) + rank(abs_(correlation(vwap, adv(20), 6))) + 0.6 * rank((ts_sum(close, 200) / 200.0 - open_) * (close - open_))",
    37: "rank(correlation(delay(open_ - close, 1), close, 200)) + rank(open_ - close)",
    38: "-rank(ts_rank(close, 10)) * rank(safe_divide(close, open_))",
    39: "-rank(delta(close, 7) * (1.0 - rank(decay_linear(safe_divide(volume, adv(20)), 9)))) * (1.0 + rank(ts_sum(returns, 250)))",
    40: "-rank(stddev(high, 10)) * correlation(high, volume, 10)",
    41: "np.sqrt(high * low) - vwap",
    42: "safe_divide(rank(vwap - close), rank(vwap + close))",
    43: "ts_rank(safe_divide(volume, adv(20)), 20) * ts_rank(-delta(close, 7), 8)",
    44: "-correlation(high, rank(volume), 5)",
    45: "-rank(ts_sum(delay(close, 5), 20) / 20.0) * correlation(close, volume, 2) * rank(correlation(ts_sum(close, 5), ts_sum(close, 20), 2))",
    46: "where(trend_acceleration(close) > 0.25, -1.0, where(trend_acceleration(close) < 0, 1.0, -delta(close, 1)))",
    47: "safe_divide(rank(safe_divide(1.0, close)) * volume, adv(20)) * safe_divide(high * rank(high - close), ts_sum(high, 5) / 5.0) - rank(vwap - delay(vwap, 5))",
    49: "where(trend_acceleration(close) < -0.1, 1.0, -delta(close, 1))",
    50: "-ts_max(rank(correlation(rank(volume), rank(vwap), 5)), 5)",
    51: "where(trend_acceleration(close) < -0.05, 1.0, -delta(close, 1))",
    52: "(-ts_min(low, 5) + delay(ts_min(low, 5), 5)) * rank((ts_sum(returns, 240) - ts_sum(returns, 20)) / 220.0) * ts_rank(volume, 5)",
    53: "-delta(safe_divide((close - low) - (high - close), close - low), 9)",
    54: "safe_divide(-(low - close) * (open_ ** 5), (low - high) * (close ** 5))",
    55: "-correlation(rank(safe_divide(close - ts_min(low, 12), ts_max(high, 12) - ts_min(low, 12))), rank(volume), 6)",
    56: "-rank(safe_divide(ts_sum(returns, 10), ts_sum(ts_sum(returns, 2), 3))) * rank(returns * cap)",
    57: "-safe_divide(close - vwap, decay_linear(rank(ts_argmax(close, 30)), 2))",
    58: "-ts_rank(decay_linear(correlation(neutralize(vwap, sector), volume, 3.92795), 7.89291), 5.50322)",
    59: "-ts_rank(decay_linear(correlation(neutralize(vwap * 0.728317 + vwap * (1 - 0.728317), industry), volume, 4.25197), 16.2289), 8.19648)",
    60: "-(2.0 * scale(rank(safe_divide(((close - low) - (high - close)) * volume, high - low))) - scale(rank(ts_argmax(close, 10))))",
    61: "(rank(vwap - ts_min(vwap, 16.1219)) < rank(correlation(vwap, adv(180), 17.9282))).astype(float)",
    62: "((rank(correlation(vwap, ts_sum(adv(20), 22.4101), 9.91009)) < rank((rank(open_) + rank(open_)) < (rank((high + low) / 2.0) + rank(high)))) * -1).astype(float)",
    63: "-(rank(decay_linear(delta(neutralize(close, industry), 2.25164), 8.22237)) - rank(decay_linear(correlation(vwap * 0.318108 + open_ * (1 - 0.318108), ts_sum(adv(180), 37.2467), 13.557), 12.2883)))",
    64: "((rank(correlation(ts_sum(open_ * 0.178404 + low * (1 - 0.178404), 12.7054), ts_sum(adv(120), 12.7054), 16.6208)) < rank(delta(((high + low) / 2.0) * 0.178404 + vwap * (1 - 0.178404), 3.69741))) * -1).astype(float)",
    65: "((rank(correlation(open_ * 0.00817205 + vwap * (1 - 0.00817205), ts_sum(adv(60), 8.6911), 6.40374)) < rank(open_ - ts_min(open_, 13.635))) * -1).astype(float)",
    66: "-(rank(decay_linear(delta(vwap, 3.51013), 7.23052)) + ts_rank(decay_linear(safe_divide((low * 0.96633 + low * (1 - 0.96633)) - vwap, open_ - (high + low) / 2.0), 11.4157), 6.72611))",
    68: "((ts_rank(correlation(rank(high), rank(adv(15)), 8.91644), 13.9333) < rank(delta(close * 0.518371 + low * (1 - 0.518371), 1.06157))) * -1).astype(float)",
    69: "-signed_power(rank(ts_max(delta(neutralize(vwap, industry), 2.72412), 4.79344)), ts_rank(correlation(close * 0.490655 + vwap * (1 - 0.490655), adv(20), 4.92416), 9.0615))",
    70: "-signed_power(rank(delta(vwap, 1.29456)), ts_rank(correlation(neutralize(close, industry), adv(50), 17.8256), 17.9171))",
    71: "maximum(ts_rank(decay_linear(correlation(ts_rank(close, 3.43976), ts_rank(adv(180), 12.0647), 18.0175), 4.20501), 15.6948), ts_rank(decay_linear(rank((low + open_) - (vwap + vwap)) ** 2, 16.4662), 4.4388))",
    72: "safe_divide(rank(decay_linear(correlation((high + low) / 2.0, adv(40), 8.93345), 10.1519)), rank(decay_linear(correlation(ts_rank(vwap, 3.72469), ts_rank(volume, 18.5188), 6.86671), 2.95011)))",
    73: "-maximum(rank(decay_linear(delta(vwap, 4.72775), 2.91864)), ts_rank(decay_linear(-safe_divide(delta(open_ * 0.147155 + low * (1 - 0.147155), 2.03608), open_ * 0.147155 + low * (1 - 0.147155)), 3.33829), 16.7411))",
    74: "((rank(correlation(close, ts_sum(adv(30), 37.4843), 15.1365)) < rank(correlation(rank(high * 0.0261661 + vwap * (1 - 0.0261661)), rank(volume), 11.4791))) * -1).astype(float)",
    75: "(rank(correlation(vwap, volume, 4.24304)) < rank(correlation(rank(low), rank(adv(50)), 12.4413))).astype(float)",
    76: "-maximum(rank(decay_linear(delta(vwap, 1.24383), 11.8259)), ts_rank(decay_linear(ts_rank(correlation(neutralize(low, sector), adv(81), 8.14941), 19.569), 17.1543), 19.383))",
    77: "minimum(rank(decay_linear((((high + low) / 2.0 + high) - (vwap + high)), 20.0451)), rank(decay_linear(correlation((high + low) / 2.0, adv(40), 3.1614), 5.64125)))",
    78: "signed_power(rank(correlation(ts_sum(low * 0.352233 + vwap * (1 - 0.352233), 19.7428), ts_sum(adv(40), 19.7428), 6.83313)), rank(correlation(rank(vwap), rank(volume), 5.77492)))",
    79: "(rank(delta(neutralize(close * 0.60733 + open_ * (1 - 0.60733), sector), 1.23438)) < rank(correlation(ts_rank(vwap, 3.60973), ts_rank(adv(150), 9.18637), 14.6644))).astype(float)",
    80: "-signed_power(rank(sign(delta(neutralize(open_ * 0.868128 + high * (1 - 0.868128), industry), 4.04545))), ts_rank(correlation(high, adv(10), 5.11456), 5.53756))",
    81: "((rank(log(ts_product(rank(rank(correlation(vwap, ts_sum(adv(10), 49.6054), 8.47743)) ** 4), 14.9655))) < rank(correlation(rank(vwap), rank(volume), 5.07914))) * -1).astype(float)",
    82: "-minimum(rank(decay_linear(delta(open_, 1.46063), 14.8717)), ts_rank(decay_linear(correlation(neutralize(volume, sector), open_ * 0.634196 + open_ * (1 - 0.634196), 17.4842), 6.92131), 13.4283))",
    83: "safe_divide(rank(delay(safe_divide(high - low, ts_sum(close, 5) / 5.0), 2)) * rank(rank(volume)), safe_divide(safe_divide(high - low, ts_sum(close, 5) / 5.0), vwap - close))",
    84: "signed_power(ts_rank(vwap - ts_max(vwap, 15.3217), 20.7127), delta(close, 4.96796))",
    85: "signed_power(rank(correlation(high * 0.876703 + close * (1 - 0.876703), adv(30), 9.61331)), rank(correlation(ts_rank((high + low) / 2.0, 3.70596), ts_rank(volume, 10.1595), 7.11408)))",
    86: "((ts_rank(correlation(close, ts_sum(adv(20), 14.7444), 6.00049), 20.4195) < rank((open_ + close) - (vwap + open_))) * -1).astype(float)",
    87: "-maximum(rank(decay_linear(delta(close * 0.369701 + vwap * (1 - 0.369701), 1.91233), 2.65461)), ts_rank(decay_linear(abs_(correlation(neutralize(adv(81), industry), close, 13.4132)), 4.89768), 14.4535))",
    88: "minimum(rank(decay_linear((rank(open_) + rank(low)) - (rank(high) + rank(close)), 8.06882)), ts_rank(decay_linear(correlation(ts_rank(close, 8.44728), ts_rank(adv(60), 20.6966), 8.01266), 6.65053), 2.61957))",
    89: "ts_rank(decay_linear(correlation(low * 0.967285 + low * (1 - 0.967285), adv(10), 6.94279), 5.51607), 3.79744) - ts_rank(decay_linear(delta(neutralize(vwap, industry), 3.48158), 10.1466), 15.3012)",
    91: "-(ts_rank(decay_linear(decay_linear(correlation(neutralize(close, industry), volume, 9.74928), 16.398), 3.83219), 4.8667) - rank(decay_linear(correlation(vwap, adv(30), 4.01303), 2.6809)))",
    92: "minimum(ts_rank(decay_linear((((high + low) / 2.0 + close) < (low + open_)).astype(float), 14.7221), 18.8683), ts_rank(decay_linear(correlation(rank(low), rank(adv(30)), 7.58555), 6.94024), 6.80584))",
    93: "safe_divide(ts_rank(decay_linear(correlation(neutralize(vwap, industry), adv(81), 17.4193), 19.848), 7.54455), rank(decay_linear(delta(close * 0.524434 + vwap * (1 - 0.524434), 2.77377), 16.2664)))",
    94: "-signed_power(rank(vwap - ts_min(vwap, 11.5783)), ts_rank(correlation(ts_rank(vwap, 19.6462), ts_rank(adv(60), 4.02992), 18.0926), 2.70756))",
    95: "(rank(open_ - ts_min(open_, 12.4105)) < ts_rank(rank(correlation(ts_sum((high + low) / 2.0, 19.1351), ts_sum(adv(40), 19.1351), 12.8742)) ** 5, 11.7584)).astype(float)",
    96: "-maximum(ts_rank(decay_linear(correlation(rank(vwap), rank(volume), 3.83878), 4.16783), 8.38151), ts_rank(decay_linear(ts_argmax(correlation(ts_rank(close, 7.45404), ts_rank(adv(60), 4.13242), 3.65459), 12.6556), 14.0365), 13.4143))",
    97: "-(rank(decay_linear(delta(neutralize(low * 0.721001 + vwap * (1 - 0.721001), industry), 3.3705), 20.4523)) - ts_rank(decay_linear(ts_rank(correlation(ts_rank(low, 7.87871), ts_rank(adv(60), 17.255), 4.97547), 18.5925), 15.7152), 6.71659))",
    98: "rank(decay_linear(correlation(vwap, ts_sum(adv(5), 26.4719), 4.58418), 7.18088)) - rank(decay_linear(ts_rank(ts_argmin(correlation(rank(open_), rank(adv(15)), 20.8187), 8.62571), 6.95668), 8.07206))",
    99: "((rank(correlation(ts_sum((high + low) / 2.0, 19.8975), ts_sum(adv(60), 19.8975), 8.8136)) < rank(correlation(low, volume, 6.28259))) * -1).astype(float)",
    101: "safe_divide(close - open_, (high - low) + 0.001)",
}


@dataclass
class AlphaPanel:
    open: pd.DataFrame
    high: pd.DataFrame
    low: pd.DataFrame
    close: pd.DataFrame
    volume: pd.DataFrame
    amount: pd.DataFrame
    returns: pd.DataFrame
    vwap: pd.DataFrame
    cap: pd.DataFrame
    sector: pd.DataFrame
    industry: pd.DataFrame


class Alpha101RESSET:
    def __init__(self, panel: AlphaPanel):
        self.panel = panel

    def _environment(self):
        p = self.panel
        return {
            "np": np,
            "float": float,
            "open_": p.open,
            "high": p.high,
            "low": p.low,
            "close": p.close,
            "volume": p.volume,
            "amount": p.amount,
            "returns": p.returns,
            "vwap": p.vwap,
            "cap": p.cap,
            "sector": p.sector,
            "industry": p.industry,
            "adv": lambda days: ts_mean(p.amount, days),
            "rank": rank,
            "delay": delay,
            "delta": delta,
            "ts_sum": ts_sum,
            "ts_mean": ts_mean,
            "stddev": stddev,
            "correlation": correlation,
            "covariance": covariance,
            "ts_min": ts_min,
            "ts_max": ts_max,
            "ts_product": ts_product,
            "ts_rank": ts_rank,
            "ts_argmax": ts_argmax,
            "ts_argmin": ts_argmin,
            "decay_linear": decay_linear,
            "scale": scale,
            "signed_power": signed_power,
            "safe_divide": safe_divide,
            "where": where,
            "minimum": minimum,
            "maximum": maximum,
            "neutralize": neutralize,
            "trend_acceleration": trend_acceleration,
            "abs_": lambda x: x.abs(),
            "sign": lambda x: pd.DataFrame(np.sign(x), index=x.index, columns=x.columns),
            "log": lambda x: np.log(x.where(x > 0)),
        }

    def compute(self, number: int) -> pd.DataFrame:
        if number in UNAVAILABLE_REASONS:
            return pd.DataFrame(np.nan, index=self.panel.close.index, columns=self.panel.close.columns)
        expression = PYTHON_EXPRESSIONS[number]
        result = eval(expression, {"__builtins__": {}}, self._environment())
        if not isinstance(result, pd.DataFrame):
            raise TypeError(f"Alpha{number:03d} 未返回DataFrame，而是 {type(result)}")
        return _clean(result).reindex_like(self.panel.close)


In [3]:

DATA_PATH = Path(r"D:\因子分析\数据\RESSET_DRESSTK_2016_2025.parquet")

# 默认执行的是实际RESSET数据上的有界烟雾测试。设为False并提供UNIVERSE_CODES可扩展到研究样本。
SMOKE_TEST = True
DATE_START = "2016-01-01"
DATE_END = "2020-12-31"
MAX_STOCKS = 60 if SMOKE_TEST else None
UNIVERSE_CODES = None  # 例如 ["000001", "000002", ...]；完整研究应传入点时点股票池。
PRICE_MODE = "forward_adjusted"  # "forward_adjusted"使用Mcfacpr；"raw"使用原始OHLC。

FIELD_MAP = {
    "code": "股票代码_Stkcd",
    "date": "日期_Date",
    "sector": "证监会行业门类代码_Csrciccd1",
    "industry": "证监会行业大类代码_Csrciccd2",
    "open": "开盘价(元)_Oppr",
    "high": "最高价(元)_Hipr",
    "low": "最低价(元)_Lopr",
    "close": "收盘价(元)_Clpr",
    "volume": "成交量(股)_Trdvol",
    "amount": "成交金额(元)_Trdsum",
    "returns": "日收益率_Dret",
    "shares": "总股数(股)_Fullshr",
    "adjust_factor": "累积股价调整乘子_Mcfacpr",
}


def load_resset_panel(path=DATA_PATH) -> tuple[AlphaPanel, pd.DataFrame]:
    columns = list(dict.fromkeys(FIELD_MAP.values()))
    filters = [
        (FIELD_MAP["date"], ">=", pd.Timestamp(DATE_START)),
        (FIELD_MAP["date"], "<=", pd.Timestamp(DATE_END)),
    ]
    raw = pd.read_parquet(path, columns=columns, filters=filters)
    raw[FIELD_MAP["date"]] = pd.to_datetime(raw[FIELD_MAP["date"]])
    raw[FIELD_MAP["code"]] = raw[FIELD_MAP["code"]].astype(str).str.zfill(6)
    raw = raw.sort_values([FIELD_MAP["code"], FIELD_MAP["date"]]).drop_duplicates(
        [FIELD_MAP["code"], FIELD_MAP["date"]], keep="last"
    )

    trading = (raw[FIELD_MAP["volume"]] > 0) & (raw[FIELD_MAP["amount"]] > 0)
    trading_dates = pd.DatetimeIndex(sorted(raw.loc[trading, FIELD_MAP["date"]].unique()))

    if UNIVERSE_CODES is not None:
        selected = [str(code).zfill(6) for code in UNIVERSE_CODES]
    elif MAX_STOCKS is not None:
        completeness = raw.loc[trading].groupby(FIELD_MAP["code"])[FIELD_MAP["date"]].nunique()
        selected = completeness.nlargest(MAX_STOCKS).index.tolist()
    else:
        selected = sorted(raw[FIELD_MAP["code"]].unique())

    data = raw.loc[raw[FIELD_MAP["code"]].isin(selected)].copy()
    data[FIELD_MAP["shares"]] = data.groupby(FIELD_MAP["code"], sort=False)[FIELD_MAP["shares"]].ffill()
    data[FIELD_MAP["adjust_factor"]] = data.groupby(FIELD_MAP["code"], sort=False)[FIELD_MAP["adjust_factor"]].ffill()
    data[FIELD_MAP["sector"]] = data.groupby(FIELD_MAP["code"], sort=False)[FIELD_MAP["sector"]].ffill()
    data[FIELD_MAP["industry"]] = data.groupby(FIELD_MAP["code"], sort=False)[FIELD_MAP["industry"]].ffill()

    valid_trade = (data[FIELD_MAP["volume"]] > 0) & (data[FIELD_MAP["amount"]] > 0)
    for key in ["open", "high", "low", "close", "volume", "amount", "returns"]:
        data.loc[~valid_trade, FIELD_MAP[key]] = np.nan

    if PRICE_MODE == "forward_adjusted":
        adjustment = data[FIELD_MAP["adjust_factor"]]
    elif PRICE_MODE == "raw":
        adjustment = 1.0
    else:
        raise ValueError("PRICE_MODE只能是'forward_adjusted'或'raw'")

    for key in ["open", "high", "low", "close"]:
        data[f"_{key}"] = data[FIELD_MAP[key]] * adjustment
    data["_volume"] = data[FIELD_MAP["volume"]]
    data["_amount"] = data[FIELD_MAP["amount"]]
    data["_returns"] = data[FIELD_MAP["returns"]]
    data["_vwap"] = safe_divide(data[FIELD_MAP["amount"]], data[FIELD_MAP["volume"]]) * adjustment
    data["_cap"] = data[FIELD_MAP["close"]] * data[FIELD_MAP["shares"]]

    def pivot(column):
        return data.pivot(index=FIELD_MAP["date"], columns=FIELD_MAP["code"], values=column).reindex(
            index=trading_dates, columns=selected
        )

    panel = AlphaPanel(
        open=pivot("_open"),
        high=pivot("_high"),
        low=pivot("_low"),
        close=pivot("_close"),
        volume=pivot("_volume"),
        amount=pivot("_amount"),
        returns=pivot("_returns"),
        vwap=pivot("_vwap"),
        cap=pivot("_cap"),
        sector=pivot(FIELD_MAP["sector"]),
        industry=pivot(FIELD_MAP["industry"]),
    )
    audit = pd.DataFrame({
        "指标": ["起始交易日", "结束交易日", "交易日数", "股票数", "价格模式", "未来回填"],
        "值": [panel.close.index.min().date(), panel.close.index.max().date(), len(panel.close), len(panel.close.columns), PRICE_MODE, "未使用"],
    })
    return panel, audit


panel, data_audit = load_resset_panel()
data_audit


,指标,值
0,起始交易日,2016-01-04
1,结束交易日,2020-12-31
2,交易日数,1218
3,股票数,60
4,价格模式,forward_adjusted
5,未来回填,未使用


## Alpha formulas, definitions, and implementation status

每个条目按“原公式 - 文字定义 - 状态/原因”的顺序列出。`PROXY_CSRC`表示结构上可计算，但行业分类口径是适配代理。

### Alpha001

**原公式：**

```text
(rank(Ts_ArgMax(SignedPower(((returns < 0) ? stddev(returns, 20) : close), 2.), 5)) -0.5)
```

**文字定义：** 当收益率为负时用20日收益波动率替代收盘价，否则使用收盘价；平方后寻找近5日最大值出现位置，再做当日截面排名并减0.5。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha002

**原公式：**

```text
(-1 * correlation(rank(delta(log(volume), 2)), rank(((close - open) / open)), 6))
```

**文字定义：** 衡量近6日内，对数成交量的2日变化与开盘到收盘日内收益之间的排名相关性，并取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha003

**原公式：**

```text
(-1 * correlation(rank(open), rank(volume), 10))
```

**文字定义：** 计算开盘价截面排名与成交量截面排名在近10日的相关性，并取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha004

**原公式：**

```text
(-1 * Ts_Rank(rank(low), 9))
```

**文字定义：** 先对最低价做每日截面排名，再计算其近9日时序排名并取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha005

**原公式：**

```text
(rank((open - (sum(vwap, 10) / 10))) * (-1 * abs(rank((close - vwap)))))
```

**文字定义：** 把开盘价相对10日VWAP均值的偏离排名，与收盘价相对VWAP偏离排名的负绝对值相乘。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha006

**原公式：**

```text
(-1 * correlation(open, volume, 10))
```

**文字定义：** 计算开盘价与成交量在近10日的时序相关性并取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha007

**原公式：**

```text
((adv20 < volume) ? ((-1 * ts_rank(abs(delta(close, 7)), 60)) * sign(delta(close, 7))) : (-1* 1))
```

**文字定义：** 仅在当日成交量高于20日平均成交额代理条件时，用7日价格变化的方向乘以其60日时序排名；否则取-1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha008

**原公式：**

```text
(-1 * rank(((sum(open, 5) * sum(returns, 5)) - delay((sum(open, 5) * sum(returns, 5)),10))))
```

**文字定义：** 将5日开盘价总和与5日收益率总和的乘积，相对10日前的变化做截面排名并取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha009

**原公式：**

```text
((0 < ts_min(delta(close, 1), 5)) ? delta(close, 1) : ((ts_max(delta(close, 1), 5) < 0) ?delta(close, 1) : (-1 * delta(close, 1))))
```

**文字定义：** 若近5日收盘价日变化始终同号，则保留当日变化；否则反转该变化。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha010

**原公式：**

```text
rank(((0 < ts_min(delta(close, 1), 4)) ? delta(close, 1) : ((ts_max(delta(close, 1), 4) < 0)? delta(close, 1) : (-1 * delta(close, 1)))))
```

**文字定义：** 与Alpha9类似，但使用4日窗口，并对条件结果做每日截面排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha011

**原公式：**

```text
((rank(ts_max((vwap - close), 3)) + rank(ts_min((vwap - close), 3))) *rank(delta(volume, 3)))
```

**文字定义：** 将VWAP减收盘价的3日最大值排名和3日最小值排名相加，再乘以成交量3日变化排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha012

**原公式：**

```text
(sign(delta(volume, 1)) * (-1 * delta(close, 1)))
```

**文字定义：** 成交量日变化的符号，乘以收盘价日变化的相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha013

**原公式：**

```text
(-1 * rank(covariance(rank(close), rank(volume), 5)))
```

**文字定义：** 计算收盘价排名与成交量排名的5日协方差，对其做截面排名后取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha014

**原公式：**

```text
((-1 * rank(delta(returns, 3))) * correlation(open, volume, 10))
```

**文字定义：** 3日收益率变化的负截面排名，乘以开盘价与成交量的10日相关性。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha015

**原公式：**

```text
(-1 * sum(rank(correlation(rank(high), rank(volume), 3)), 3))
```

**文字定义：** 计算最高价排名和成交量排名的3日相关性，对相关性排名后求3日和并取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha016

**原公式：**

```text
(-1 * rank(covariance(rank(high), rank(volume), 5)))
```

**文字定义：** 计算最高价排名与成交量排名的5日协方差，做截面排名后取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha017

**原公式：**

```text
(((-1 * rank(ts_rank(close, 10))) * rank(delta(delta(close, 1), 1))) *rank(ts_rank((volume / adv20), 5)))
```

**文字定义：** 把收盘价10日时序排名、价格变化的二阶差分以及量能相对ADV20的5日时序排名组合为负向乘积。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha018

**原公式：**

```text
(-1 * rank(((stddev(abs((close - open)), 5) + (close - open)) + correlation(close, open,10))))
```

**文字定义：** 把开收盘价差的5日波动、当日开收盘价差及开盘价与收盘价10日相关性相加，排名后取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha019

**原公式：**

```text
((-1 * sign(((close - delay(close, 7)) + delta(close, 7)))) * (1 + rank((1 + sum(returns,250)))))
```

**文字定义：** 以7日价格变化方向构造反转信号，并用250日累计收益的截面排名调整强度。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha020

**原公式：**

```text
(((-1 * rank((open - delay(high, 1)))) * rank((open - delay(close, 1)))) * rank((open -delay(low, 1))))
```

**文字定义：** 比较今日开盘价与昨日最高、收盘、最低价的差异，分别排名后相乘并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha021

**原公式：**

```text
((((sum(close, 8) / 8) + stddev(close, 8)) < (sum(close, 2) / 2)) ? (-1 * 1) : (((sum(close,2) / 2) < ((sum(close, 8) / 8) - stddev(close, 8))) ? 1 : (((1 < (volume / adv20)) || ((volume /adv20) == 1)) ? 1 : (-1 * 1))))
```

**文字定义：** 根据短期均价相对8日均值加减波动带的位置给出-1或1；若不触发价格条件，再由量能相对ADV20决定。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha022

**原公式：**

```text
(-1 * (delta(correlation(high, volume, 5), 5) * rank(stddev(close, 20))))
```

**文字定义：** 最高价与成交量5日相关性的5日变化，乘以收盘价20日波动率排名并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha023

**原公式：**

```text
(((sum(high, 20) / 20) < high) ? (-1 * delta(high, 2)) : 0)
```

**文字定义：** 当最高价高于其20日均值时，取最高价2日变化的相反数，否则为0。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha024

**原公式：**

```text
((((delta((sum(close, 100) / 100), 100) / delay(close, 100)) < 0.05) ||((delta((sum(close, 100) / 100), 100) / delay(close, 100)) == 0.05)) ? (-1 * (close - ts_min(close,100))) : (-1 * delta(close, 3)))
```

**文字定义：** 当100日均价趋势相对100日前收盘价不超过5%时使用距100日最低价的负偏离，否则使用3日价格变化的相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha025

**原公式：**

```text
rank(((((-1 * returns) * adv20) * vwap) * (high - close)))
```

**文字定义：** 将负收益、ADV20、VWAP以及最高价减收盘价的乘积做每日截面排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha026

**原公式：**

```text
(-1 * ts_max(correlation(ts_rank(volume, 5), ts_rank(high, 5), 5), 3))
```

**文字定义：** 成交量5日时序排名与最高价5日时序排名的5日相关性，取其近3日最大值并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha027

**原公式：**

```text
((0.5 < rank((sum(correlation(rank(volume), rank(vwap), 6), 2) / 2.0))) ? (-1 * 1) : 1)
```

**文字定义：** 将成交量排名和VWAP排名的6日相关性做2日平均及截面排名，高于0.5取-1，否则取1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha028

**原公式：**

```text
scale(((correlation(adv20, low, 5) + ((high + low) / 2)) - close))
```

**文字定义：** 把ADV20与最低价的5日相关性、日内中间价和收盘价组合后，按日做绝对值和为1的截面缩放。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha029

**原公式：**

```text
(min(product(rank(rank(scale(log(sum(ts_min(rank(rank((-1 * rank(delta((close - 1),5))))), 2), 1))))), 1), 5) + ts_rank(delay((-1 * returns), 6), 5))
```

**文字定义：** 对价格变化排名进行多层排名、缩放、对数和短窗极值处理，再加上滞后负收益的5日时序排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha030

**原公式：**

```text
(((1.0 - rank(((sign((close - delay(close, 1))) + sign((delay(close, 1) - delay(close, 2)))) +sign((delay(close, 2) - delay(close, 3)))))) * sum(volume, 5)) / sum(volume, 20))
```

**文字定义：** 用最近三次价格变化方向之和的截面排名构造反转项，再乘以5日成交量总和相对20日总和的比例。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha031

**原公式：**

```text
((rank(rank(rank(decay_linear((-1 * rank(rank(delta(close, 10)))), 10)))) + rank((-1 *delta(close, 3)))) + sign(scale(correlation(adv20, low, 12))))
```

**文字定义：** 组合10日价格变化的线性衰减多重排名、3日价格变化负排名，以及ADV20和最低价相关性的方向。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha032

**原公式：**

```text
(scale(((sum(close, 7) / 7) - close)) + (20 * scale(correlation(vwap, delay(close, 5),230))))
```

**文字定义：** 组合收盘价相对7日均价的截面缩放项，以及VWAP与滞后5日收盘价的230日相关性缩放项。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha033

**原公式：**

```text
rank((-1 * ((1 - (open / close))^1)))
```

**文字定义：** 对开盘价相对收盘价的比例减1做每日截面排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha034

**原公式：**

```text
rank(((1 - rank((stddev(returns, 2) / stddev(returns, 5)))) + (1 - rank(delta(close, 1)))))
```

**文字定义：** 结合2日与5日收益波动率之比的反排名，以及收盘价日变化的反排名，再做截面排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha035

**原公式：**

```text
((Ts_Rank(volume, 32) * (1 - Ts_Rank(((close + high) - low), 16))) * (1 -Ts_Rank(returns, 32)))
```

**文字定义：** 将成交量32日时序排名、价格区间组合的16日反时序排名和收益率32日反时序排名相乘。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha036

**原公式：**

```text
(((((2.21 * rank(correlation((close - open), delay(volume, 1), 15))) + (0.7 * rank((open- close)))) + (0.73 * rank(Ts_Rank(delay((-1 * returns), 6), 5)))) + rank(abs(correlation(vwap,adv20, 6)))) + (0.6 * rank((((sum(close, 200) / 200) - open) * (close - open)))))
```

**文字定义：** 按给定权重组合开收盘差与滞后成交量相关性、开收盘差、滞后负收益排名、VWAP与ADV20相关性及长期均价偏离。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha037

**原公式：**

```text
(rank(correlation(delay((open - close), 1), close, 200)) + rank((open - close)))
```

**文字定义：** 把昨日开收盘差与收盘价的200日相关性排名，加上当日开收盘差排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha038

**原公式：**

```text
((-1 * rank(Ts_Rank(close, 10))) * rank((close / open)))
```

**文字定义：** 收盘价10日时序排名的负截面排名，乘以收盘价相对开盘价比例的截面排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha039

**原公式：**

```text
((-1 * rank((delta(close, 7) * (1 - rank(decay_linear((volume / adv20), 9)))))) * (1 +rank(sum(returns, 250))))
```

**文字定义：** 以7日价格变化和量能相对ADV20的线性衰减排名构造反向项，再由250日累计收益排名放大。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha040

**原公式：**

```text
((-1 * rank(stddev(high, 10))) * correlation(high, volume, 10))
```

**文字定义：** 最高价10日波动率的负截面排名，乘以最高价与成交量的10日相关性。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha041

**原公式：**

```text
(((high * low)^0.5) - vwap)
```

**文字定义：** 最高价与最低价几何平均值减去VWAP。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha042

**原公式：**

```text
(rank((vwap - close)) / rank((vwap + close)))
```

**文字定义：** VWAP减收盘价的截面排名，除以VWAP加收盘价的截面排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha043

**原公式：**

```text
(ts_rank((volume / adv20), 20) * ts_rank((-1 * delta(close, 7)), 8))
```

**文字定义：** 量能相对ADV20的20日时序排名，乘以负7日价格变化的8日时序排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha044

**原公式：**

```text
(-1 * correlation(high, rank(volume), 5))
```

**文字定义：** 最高价与成交量截面排名的5日相关性，并取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha045

**原公式：**

```text
(-1 * ((rank((sum(delay(close, 5), 20) / 20)) * correlation(close, volume, 2)) *rank(correlation(sum(close, 5), sum(close, 20), 2))))
```

**文字定义：** 组合滞后收盘价20日均值排名、收盘价与成交量2日相关性，以及5日与20日价格总和相关性的排名，并整体取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha046

**原公式：**

```text
((0.25 < (((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10))) ?(-1 * 1) : (((((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10)) < 0) ? 1 :((-1 * 1) * (close - delay(close, 1)))))
```

**文字定义：** 根据10日与20日价格趋势差判断：高于0.25取-1，低于0取1，否则取当日价格变化的相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha047

**原公式：**

```text
((((rank((1 / close)) * volume) / adv20) * ((high * rank((high - close))) / (sum(high, 5) /5))) - rank((vwap - delay(vwap, 5))))
```

**文字定义：** 结合低价股排名、成交量相对ADV20、最高价位置，并减去VWAP相对5日前变化的排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha048

**原公式：**

```text
(indneutralize(((correlation(delta(close, 1), delta(delay(close, 1), 1), 250) *delta(close, 1)) / close), IndClass.subindustry) / sum(((delta(close, 1) / delay(close, 1))^2), 250))
```

**文字定义：** 将价格变化自相关相关项按细分行业中性化，再除以250日收益变化平方和。

**实现状态：** `UNAVAILABLE`

**说明：** 需要逐日历史 IndClass.subindustry；当前RESSET文件只有证监会一级和二级行业，不能严格完成细分行业中性化。

### Alpha049

**原公式：**

```text
(((((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10)) < (-1 *0.1)) ? 1 : ((-1 * 1) * (close - delay(close, 1))))
```

**文字定义：** 当10日与20日价格趋势差低于-0.1时取1，否则取当日价格变化的相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha050

**原公式：**

```text
(-1 * ts_max(rank(correlation(rank(volume), rank(vwap), 5)), 5))
```

**文字定义：** 成交量排名与VWAP排名5日相关性的截面排名，取近5日最大值并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha051

**原公式：**

```text
(((((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10)) < (-1 *0.05)) ? 1 : ((-1 * 1) * (close - delay(close, 1))))
```

**文字定义：** 与Alpha49相同，但趋势触发阈值改为-0.05。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha052

**原公式：**

```text
((((-1 * ts_min(low, 5)) + delay(ts_min(low, 5), 5)) * rank(((sum(returns, 240) -sum(returns, 20)) / 220))) * ts_rank(volume, 5))
```

**文字定义：** 结合5日最低价的5日反向变化、240日与20日累计收益差的排名，以及成交量5日时序排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha053

**原公式：**

```text
(-1 * delta((((close - low) - (high - close)) / (close - low)), 9))
```

**文字定义：** 对K线收盘位置比率做9日变化并取相反数。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha054

**原公式：**

```text
((-1 * ((low - close) * (open^5))) / ((low - high) * (close^5)))
```

**文字定义：** 用开盘价和收盘价的五次幂，结合最低价与收盘价、最高价的距离构造非线性日内信号。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha055

**原公式：**

```text
(-1 * correlation(rank(((close - ts_min(low, 12)) / (ts_max(high, 12) - ts_min(low,12)))), rank(volume), 6))
```

**文字定义：** 收盘价在12日高低区间中的位置排名，与成交量排名的6日相关性取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha056

**原公式：**

```text
(0 - (1 * (rank((sum(returns, 10) / sum(sum(returns, 2), 3))) * rank((returns * cap)))))
```

**文字定义：** 10日收益总和相对嵌套短期收益总和的排名，乘以收益率与总市值乘积的排名并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha057

**原公式：**

```text
(0 - (1 * ((close - vwap) / decay_linear(rank(ts_argmax(close, 30)), 2))))
```

**文字定义：** 收盘价减VWAP，除以收盘价30日最高值出现位置排名的2日线性衰减值，并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha058

**原公式：**

```text
(-1 * Ts_Rank(decay_linear(correlation(IndNeutralize(vwap, IndClass.sector), volume,3.92795), 7.89291), 5.50322))
```

**文字定义：** 对行业板块中性化后的VWAP与成交量做短期相关、线性衰减及时序排名，并取负；本适配版以证监会一级行业代理sector。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha059

**原公式：**

```text
(-1 * Ts_Rank(decay_linear(correlation(IndNeutralize(((vwap * 0.728317) + (vwap *(1 - 0.728317))), IndClass.industry), volume, 4.25197), 16.2289), 8.19648))
```

**文字定义：** 对行业中性化VWAP与成交量的短期相关性做线性衰减和时序排名并取负；本适配版以证监会二级行业代理industry。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha060

**原公式：**

```text
(0 - (1 * ((2 * scale(rank(((((close - low) - (high - close)) / (high - low)) * volume)))) -scale(rank(ts_argmax(close, 10))))))
```

**文字定义：** 结合收盘价在日内区间的位置乘成交量的排名，以及收盘价10日最高值出现位置排名，分别缩放后相减并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha061

**原公式：**

```text
(rank((vwap - ts_min(vwap, 16.1219))) < rank(correlation(vwap, adv180, 17.9282)))
```

**文字定义：** 比较VWAP距近16日最低值的排名，与VWAP和ADV180近17日相关性的排名，输出0或1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha062

**原公式：**

```text
((rank(correlation(vwap, sum(adv20, 22.4101), 9.91009)) < rank(((rank(open) +rank(open)) < (rank(((high + low) / 2)) + rank(high))))) * -1)
```

**文字定义：** 比较VWAP与ADV20滚动总和相关性的排名，和开盘价排名相对高低价排名组合的布尔排名，条件成立时取-1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha063

**原公式：**

```text
((rank(decay_linear(delta(IndNeutralize(close, IndClass.industry), 2.25164), 8.22237))- rank(decay_linear(correlation(((vwap * 0.318108) + (open * (1 - 0.318108))), sum(adv180,37.2467), 13.557), 12.2883))) * -1)
```

**文字定义：** 比较行业中性化收盘价变化的衰减排名，与VWAP/开盘混合价同ADV180总和相关性的衰减排名；以证监会二级行业代理。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha064

**原公式：**

```text
((rank(correlation(sum(((open * 0.178404) + (low * (1 - 0.178404))), 12.7054),sum(adv120, 12.7054), 16.6208)) < rank(delta(((((high + low) / 2) * 0.178404) + (vwap * (1 -0.178404))), 3.69741))) * -1)
```

**文字定义：** 比较开盘价/最低价混合价总和与ADV120总和的相关性排名，以及中间价/VWAP混合价变化的排名，成立时取-1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha065

**原公式：**

```text
((rank(correlation(((open * 0.00817205) + (vwap * (1 - 0.00817205))), sum(adv60,8.6911), 6.40374)) < rank((open - ts_min(open, 13.635)))) * -1)
```

**文字定义：** 比较开盘价/VWAP混合价与ADV60总和相关性的排名，以及开盘价距近13日最低值的排名，成立时取-1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha066

**原公式：**

```text
((rank(decay_linear(delta(vwap, 3.51013), 7.23052)) + Ts_Rank(decay_linear(((((low* 0.96633) + (low * (1 - 0.96633))) - vwap) / (open - ((high + low) / 2))), 11.4157), 6.72611)) * -1)
```

**文字定义：** 把VWAP变化的衰减排名，与最低价相对VWAP和开盘价相对中间价之比的衰减时序排名相加并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha067

**原公式：**

```text
((rank((high - ts_min(high, 2.14593)))^rank(correlation(IndNeutralize(vwap,IndClass.sector), IndNeutralize(adv20, IndClass.subindustry), 6.02936))) * -1)
```

**文字定义：** 将最高价距短期最低值的排名，与sector中性化VWAP和subindustry中性化ADV20相关性的排名作幂运算并取负。

**实现状态：** `UNAVAILABLE`

**说明：** 同时需要 sector 与 subindustry；可代理sector，但缺少历史subindustry，因此保留空实现。

### Alpha068

**原公式：**

```text
((Ts_Rank(correlation(rank(high), rank(adv15), 8.91644), 13.9333) <rank(delta(((close * 0.518371) + (low * (1 - 0.518371))), 1.06157))) * -1)
```

**文字定义：** 比较最高价排名与ADV15排名相关性的时序排名，以及收盘/最低混合价变化的排名，成立时取-1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha069

**原公式：**

```text
((rank(ts_max(delta(IndNeutralize(vwap, IndClass.industry), 2.72412),4.79344))^Ts_Rank(correlation(((close * 0.490655) + (vwap * (1 - 0.490655))), adv20, 4.92416),9.0615)) * -1)
```

**文字定义：** 行业中性化VWAP变化的短期最大值排名，按价格混合项与ADV20相关性的时序排名做幂运算并取负。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha070

**原公式：**

```text
((rank(delta(vwap, 1.29456))^Ts_Rank(correlation(IndNeutralize(close,IndClass.industry), adv50, 17.8256), 17.9171)) * -1)
```

**文字定义：** VWAP变化排名，按行业中性化收盘价与ADV50相关性的时序排名做幂运算并取负。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha071

**原公式：**

```text
max(Ts_Rank(decay_linear(correlation(Ts_Rank(close, 3.43976), Ts_Rank(adv180,12.0647), 18.0175), 4.20501), 15.6948), Ts_Rank(decay_linear((rank(((low + open) - (vwap +vwap)))^2), 16.4662), 4.4388))
```

**文字定义：** 取两项较大者：价格时序排名与ADV180时序排名相关性的衰减时序排名；价格位置排名平方的衰减时序排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha072

**原公式：**

```text
(rank(decay_linear(correlation(((high + low) / 2), adv40, 8.93345), 10.1519)) /rank(decay_linear(correlation(Ts_Rank(vwap, 3.72469), Ts_Rank(volume, 18.5188), 6.86671),2.95011)))
```

**文字定义：** 中间价与ADV40相关性的衰减排名，除以VWAP时序排名与成交量时序排名相关性的衰减排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha073

**原公式：**

```text
(max(rank(decay_linear(delta(vwap, 4.72775), 2.91864)),Ts_Rank(decay_linear(((delta(((open * 0.147155) + (low * (1 - 0.147155))), 2.03608) / ((open *0.147155) + (low * (1 - 0.147155)))) * -1), 3.33829), 16.7411)) * -1)
```

**文字定义：** 取VWAP变化衰减排名与开盘/最低混合价相对变化衰减时序排名中的较大者，并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha074

**原公式：**

```text
((rank(correlation(close, sum(adv30, 37.4843), 15.1365)) <rank(correlation(rank(((high * 0.0261661) + (vwap * (1 - 0.0261661)))), rank(volume), 11.4791)))* -1)
```

**文字定义：** 比较收盘价与ADV30总和相关性的排名，以及最高价/VWAP混合价排名与成交量排名相关性的排名，成立时取-1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha075

**原公式：**

```text
(rank(correlation(vwap, volume, 4.24304)) < rank(correlation(rank(low), rank(adv50),12.4413)))
```

**文字定义：** 比较VWAP与成交量相关性的排名，以及最低价排名与ADV50排名相关性的排名，输出0或1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha076

**原公式：**

```text
(max(rank(decay_linear(delta(vwap, 1.24383), 11.8259)),Ts_Rank(decay_linear(Ts_Rank(correlation(IndNeutralize(low, IndClass.sector), adv81,8.14941), 19.569), 17.1543), 19.383)) * -1)
```

**文字定义：** 取VWAP变化衰减排名与sector中性化最低价同ADV81相关项的双重时序处理中的较大者并取负；sector用证监会一级行业代理。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha077

**原公式：**

```text
min(rank(decay_linear(((((high + low) / 2) + high) - (vwap + high)), 20.0451)),rank(decay_linear(correlation(((high + low) / 2), adv40, 3.1614), 5.64125)))
```

**文字定义：** 取中间价相对VWAP偏离的衰减排名，与中间价和ADV40相关性的衰减排名中的较小者。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha078

**原公式：**

```text
(rank(correlation(sum(((low * 0.352233) + (vwap * (1 - 0.352233))), 19.7428),sum(adv40, 19.7428), 6.83313))^rank(correlation(rank(vwap), rank(volume), 5.77492)))
```

**文字定义：** 价格混合项总和与ADV40总和相关性的排名，按VWAP排名与成交量排名相关性的排名做幂运算。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha079

**原公式：**

```text
(rank(delta(IndNeutralize(((close * 0.60733) + (open * (1 - 0.60733))),IndClass.sector), 1.23438)) < rank(correlation(Ts_Rank(vwap, 3.60973), Ts_Rank(adv150,9.18637), 14.6644)))
```

**文字定义：** 比较sector中性化收盘/开盘混合价变化的排名，以及VWAP和ADV150时序排名相关性的排名；sector用证监会一级行业代理。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha080

**原公式：**

```text
((rank(Sign(delta(IndNeutralize(((open * 0.868128) + (high * (1 - 0.868128))),IndClass.industry), 4.04545)))^Ts_Rank(correlation(high, adv10, 5.11456), 5.53756)) * -1)
```

**文字定义：** 行业中性化开盘/最高混合价变化方向的排名，按最高价与ADV10相关性的时序排名做幂运算并取负。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha081

**原公式：**

```text
((rank(Log(product(rank((rank(correlation(vwap, sum(adv10, 49.6054),8.47743))^4)), 14.9655))) < rank(correlation(rank(vwap), rank(volume), 5.07914))) * -1)
```

**文字定义：** 比较VWAP与ADV10总和相关项经幂、排名、连乘和对数后的排名，以及VWAP排名与成交量排名相关性的排名，成立时取-1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha082

**原公式：**

```text
(min(rank(decay_linear(delta(open, 1.46063), 14.8717)),Ts_Rank(decay_linear(correlation(IndNeutralize(volume, IndClass.sector), ((open * 0.634196) +(open * (1 - 0.634196))), 17.4842), 6.92131), 13.4283)) * -1)
```

**文字定义：** 取开盘价变化衰减排名和sector中性化成交量与开盘价相关项的衰减时序排名中的较小者并取负。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha083

**原公式：**

```text
((rank(delay(((high - low) / (sum(close, 5) / 5)), 2)) * rank(rank(volume))) / (((high -low) / (sum(close, 5) / 5)) / (vwap - close)))
```

**文字定义：** 把滞后的日内振幅相对5日均价之比的排名与成交量双重排名相乘，再除以当日振幅相对均价及VWAP收盘偏离的组合。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha084

**原公式：**

```text
SignedPower(Ts_Rank((vwap - ts_max(vwap, 15.3217)), 20.7127), delta(close,4.96796))
```

**文字定义：** VWAP距近15日最高值的20日时序排名，以收盘价4日变化为指数做有符号幂运算。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha085

**原公式：**

```text
(rank(correlation(((high * 0.876703) + (close * (1 - 0.876703))), adv30,9.61331))^rank(correlation(Ts_Rank(((high + low) / 2), 3.70596), Ts_Rank(volume, 10.1595),7.11408)))
```

**文字定义：** 最高/收盘混合价与ADV30相关性的排名，按中间价时序排名与成交量时序排名相关性的排名做幂运算。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha086

**原公式：**

```text
((Ts_Rank(correlation(close, sum(adv20, 14.7444), 6.00049), 20.4195) < rank(((open+ close) - (vwap + open)))) * -1)
```

**文字定义：** 比较收盘价与ADV20总和相关性的时序排名，以及收盘价减VWAP的截面排名，成立时取-1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha087

**原公式：**

```text
(max(rank(decay_linear(delta(((close * 0.369701) + (vwap * (1 - 0.369701))),1.91233), 2.65461)), Ts_Rank(decay_linear(abs(correlation(IndNeutralize(adv81,IndClass.industry), close, 13.4132)), 4.89768), 14.4535)) * -1)
```

**文字定义：** 取价格混合项变化的衰减排名，与行业中性化ADV81同收盘价相关性绝对值的衰减时序排名中的较大者并取负。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha088

**原公式：**

```text
min(rank(decay_linear(((rank(open) + rank(low)) - (rank(high) + rank(close))),8.06882)), Ts_Rank(decay_linear(correlation(Ts_Rank(close, 8.44728), Ts_Rank(adv60,20.6966), 8.01266), 6.65053), 2.61957))
```

**文字定义：** 取四种价格排名差的衰减排名，与收盘价和ADV60各自时序排名相关性的衰减时序排名中的较小者。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha089

**原公式：**

```text
(Ts_Rank(decay_linear(correlation(((low * 0.967285) + (low * (1 - 0.967285))), adv10,6.94279), 5.51607), 3.79744) - Ts_Rank(decay_linear(delta(IndNeutralize(vwap,IndClass.industry), 3.48158), 10.1466), 15.3012))
```

**文字定义：** 最低价与ADV10相关性的衰减时序排名，减去行业中性化VWAP变化的衰减时序排名。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha090

**原公式：**

```text
((rank((close - ts_max(close, 4.66719)))^Ts_Rank(correlation(IndNeutralize(adv40,IndClass.subindustry), low, 5.38375), 3.21856)) * -1)
```

**文字定义：** 收盘价距短期最高值的排名，按subindustry中性化ADV40与最低价相关性的时序排名做幂运算并取负。

**实现状态：** `UNAVAILABLE`

**说明：** 需要用历史subindustry对ADV40做截面中性化；当前数据缺少该层级。

### Alpha091

**原公式：**

```text
((Ts_Rank(decay_linear(decay_linear(correlation(IndNeutralize(close,IndClass.industry), volume, 9.74928), 16.398), 3.83219), 4.8667) -rank(decay_linear(correlation(vwap, adv30, 4.01303), 2.6809))) * -1)
```

**文字定义：** 行业中性化收盘价与成交量相关项经两次衰减后的时序排名，减去VWAP与ADV30相关性的衰减排名，整体取负。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha092

**原公式：**

```text
min(Ts_Rank(decay_linear(((((high + low) / 2) + close) < (low + open)), 14.7221),18.8683), Ts_Rank(decay_linear(correlation(rank(low), rank(adv30), 7.58555), 6.94024),6.80584))
```

**文字定义：** 取价格不等式布尔值的衰减时序排名，与最低价排名和ADV30排名相关性的衰减时序排名中的较小者。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha093

**原公式：**

```text
(Ts_Rank(decay_linear(correlation(IndNeutralize(vwap, IndClass.industry), adv81,17.4193), 19.848), 7.54455) / rank(decay_linear(delta(((close * 0.524434) + (vwap * (1 -0.524434))), 2.77377), 16.2664)))
```

**文字定义：** 行业中性化VWAP与ADV81相关性的衰减时序排名，除以收盘/VWAP混合价变化的衰减排名。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha094

**原公式：**

```text
((rank((vwap - ts_min(vwap, 11.5783)))^Ts_Rank(correlation(Ts_Rank(vwap,19.6462), Ts_Rank(adv60, 4.02992), 18.0926), 2.70756)) * -1)
```

**文字定义：** VWAP距近11日最低值的排名，按VWAP与ADV60时序排名相关性的时序排名做幂运算并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha095

**原公式：**

```text
(rank((open - ts_min(open, 12.4105))) < Ts_Rank((rank(correlation(sum(((high + low)/ 2), 19.1351), sum(adv40, 19.1351), 12.8742))^5), 11.7584))
```

**文字定义：** 比较开盘价距短期最低值的排名，以及中间价总和与ADV40总和相关性排名五次幂的时序排名，输出0或1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha096

**原公式：**

```text
(max(Ts_Rank(decay_linear(correlation(rank(vwap), rank(volume), 3.83878),4.16783), 8.38151), Ts_Rank(decay_linear(Ts_ArgMax(correlation(Ts_Rank(close, 7.45404),Ts_Rank(adv60, 4.13242), 3.65459), 12.6556), 14.0365), 13.4143)) * -1)
```

**文字定义：** 取VWAP排名与成交量排名相关性的衰减时序排名，和价格/ADV60时序相关性最大值位置的衰减时序排名中的较大者并取负。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha097

**原公式：**

```text
((rank(decay_linear(delta(IndNeutralize(((low * 0.721001) + (vwap * (1 - 0.721001))),IndClass.industry), 3.3705), 20.4523)) - Ts_Rank(decay_linear(Ts_Rank(correlation(Ts_Rank(low,7.87871), Ts_Rank(adv60, 17.255), 4.97547), 18.5925), 15.7152), 6.71659)) * -1)
```

**文字定义：** 比较行业中性化最低价/VWAP混合价变化的衰减排名，与最低价和ADV60多层时序相关项的衰减时序排名，整体取负。

**实现状态：** `PROXY_CSRC`

**说明：** 公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。

### Alpha098

**原公式：**

```text
(rank(decay_linear(correlation(vwap, sum(adv5, 26.4719), 4.58418), 7.18088)) -rank(decay_linear(Ts_Rank(Ts_ArgMin(correlation(rank(open), rank(adv15), 20.8187), 8.62571),6.95668), 8.07206)))
```

**文字定义：** VWAP与ADV5总和相关性的衰减排名，减去开盘价排名与ADV15排名相关性最小值位置经时序排名和衰减后的排名。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha099

**原公式：**

```text
((rank(correlation(sum(((high + low) / 2), 19.8975), sum(adv60, 19.8975), 8.8136)) <rank(correlation(low, volume, 6.28259))) * -1)
```

**文字定义：** 比较中间价总和与ADV60总和相关性的排名，以及最低价与成交量相关性的排名，成立时取-1。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

### Alpha100

**原公式：**

```text
(0 - (1 * (((1.5 * scale(indneutralize(indneutralize(rank(((((close - low) - (high -close)) / (high - low)) * volume)), IndClass.subindustry), IndClass.subindustry))) -scale(indneutralize((correlation(close, rank(adv20), 5) - rank(ts_argmin(close, 30))),IndClass.subindustry))) * (volume / adv20))))
```

**文字定义：** 将量价日内位置项两次按subindustry中性化并缩放，再减去另一subindustry中性化相关项的缩放值，最后乘量能相对ADV20并取负。

**实现状态：** `UNAVAILABLE`

**说明：** 公式两次使用subindustry中性化；当前数据缺少历史细分行业字段，不能严格实现。

### Alpha101

**原公式：**

```text
((close - open) / ((high - low) + .001))
```

**文字定义：** 收盘价减开盘价，除以当日最高价减最低价加0.001，衡量收盘在日内价格区间中的方向与幅度。

**实现状态：** `IMPLEMENTED`

**说明：** 当前RESSET字段可支持。

## Results

下面在有界真实数据样本上依次计算101个编号；不可实现因子返回同形状全NaN面板。

In [4]:

engine = Alpha101RESSET(panel)
factor_results = {}
validation_rows = []

for alpha_number in range(1, 102):
    result = engine.compute(alpha_number)
    factor_results[alpha_number] = result
    numeric = result.to_numpy(dtype=float)
    finite_count = int(np.isfinite(numeric).sum())
    validation_rows.append({
        "因子": f"Alpha{alpha_number:03d}",
        "实现状态": ALPHA_METADATA[alpha_number]["status"],
        "有限值数量": finite_count,
        "缺失率": float(np.isnan(numeric).mean()),
        "形状": f"{result.shape[0]} x {result.shape[1]}",
        "说明": ALPHA_METADATA[alpha_number]["reason"],
    })

validation = pd.DataFrame(validation_rows)
unexpected_empty = validation.loc[
    validation["实现状态"].ne("UNAVAILABLE") & validation["有限值数量"].eq(0), "因子"
].tolist()

assert len(ALPHA_METADATA) == 101
assert set(UNAVAILABLE_REASONS) == {48, 67, 90, 100}
assert not unexpected_empty, f"以下已实现因子没有产生任何有限值: {unexpected_empty}"

summary = pd.DataFrame({
    "项目": ["IMPLEMENTED", "PROXY_CSRC", "UNAVAILABLE", "已实现且产生有限值", "异常空结果"],
    "数量": [
        int((validation["实现状态"] == "IMPLEMENTED").sum()),
        int((validation["实现状态"] == "PROXY_CSRC").sum()),
        int((validation["实现状态"] == "UNAVAILABLE").sum()),
        int(((validation["实现状态"] != "UNAVAILABLE") & (validation["有限值数量"] > 0)).sum()),
        len(unexpected_empty),
    ],
})
summary


,项目,数量
0,IMPLEMENTED,83
1,PROXY_CSRC,14
2,UNAVAILABLE,4
3,已实现且产生有限值,97
4,异常空结果,0


In [5]:
validation

,因子,实现状态,有限值数量,缺失率,形状,说明
0,Alpha001,IMPLEMENTED,71717,0.018651,1218 x 60,当前RESSET字段可支持。
1,Alpha002,IMPLEMENTED,72660,0.005747,1218 x 60,当前RESSET字段可支持。
2,Alpha003,IMPLEMENTED,62422,0.145840,1218 x 60,当前RESSET字段可支持。
3,Alpha004,IMPLEMENTED,72600,0.006568,1218 x 60,当前RESSET字段可支持。
4,Alpha005,IMPLEMENTED,72540,0.007389,1218 x 60,当前RESSET字段可支持。
...,...,...,...,...,...,...
96,Alpha097,PROXY_CSRC,1349,0.981541,1218 x 60,公式可计算，但IndClass.sector/industry分别用证监会一级/二级行业代理，数值不等同于原论文分类。
97,Alpha098,IMPLEMENTED,51920,0.289546,1218 x 60,当前RESSET字段可支持。
98,Alpha099,IMPLEMENTED,73080,0.000000,1218 x 60,当前RESSET字段可支持。
99,Alpha100,UNAVAILABLE,0,1.000000,1218 x 60,公式两次使用subindustry中性化；当前数据缺少历史细分行业字段，不能严格实现。


### Checks

检查截面排名、VWAP单位、空实现以及无穷值。

In [6]:

# 边界与语义检查：截面rank、未来回填、VWAP单位、空实现。
last_complete_date = panel.close.notna().sum(axis=1).idxmax()
rank_check = rank(panel.close).loc[last_complete_date].dropna()
vwap_raw = safe_divide(panel.amount, panel.volume)

checks = pd.DataFrame({
    "检查": [
        "截面rank范围",
        "VWAP为正的比例",
        "不可实现因子是否全为空",
        "所有非缺失结果均为有限值",
    ],
    "结果": [
        bool(rank_check.between(0, 1).all()),
        float((vwap_raw > 0).stack().mean()),
        bool(all(factor_results[n].isna().all().all() for n in UNAVAILABLE_REASONS)),
        bool(all(np.isfinite(df.to_numpy(dtype=float)[~np.isnan(df.to_numpy(dtype=float))]).all() for df in factor_results.values())),
    ],
})
checks


,检查,结果
0,截面rank范围,True
1,VWAP为正的比例,1.0
2,不可实现因子是否全为空,True
3,所有非缺失结果均为有限值,True


## Takeaways

- `IMPLEMENTED`表示当前字段可直接支持，但仍需要在正式回测中加入点时点股票池和信号/成交时序。
- `PROXY_CSRC`表示仅完成证监会行业代理版本，不能声称与论文原始IndClass完全一致。
- `UNAVAILABLE`因子保持空值，避免用错误分类静默替代。
- 完整截面计算建议按因子逐个计算并落盘，避免同时保存数十个5000只股票宽表造成内存压力。

In [7]:

def export_factor_long(alpha_number: int, output_dir: Path) -> Path:
    """把一个因子的宽表结果保存成长表Parquet；全量运行时建议逐因子导出以控制内存。"""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    result = engine.compute(alpha_number)
    long_result = result.stack(dropna=False).rename("alpha_value").reset_index()
    long_result.columns = ["date", "stock_code", "alpha_value"]
    output_path = output_dir / f"alpha{alpha_number:03d}.parquet"
    long_result.to_parquet(output_path, index=False)
    return output_path


# 示例（默认不写文件）：
# export_factor_long(1, Path(r"D:\因子分析\WorldQuant_alpha101_code\alpha_outputs"))
